In [ ]:
# Environment setup


# ==================================================
# NBA DATA SCIENCE & ANALYTICS — STEP 1
# Installazione e configurazione
# ==================================================

%pip install -q "numpy==2.2.6" "pandas==2.2.3" nba_api plotly openpyxl

# Shared configuration (run this cell before the others).
# NBA_DATA_DIR can point to your own data folder; private data is never bundled.
import os
from pathlib import Path

_data_override = os.environ.get("NBA_DATA_DIR")
_mount_candidates = [
    Path("/content/drive/MyDrive/NBA_Data_Science"),
    Path("/content/gdrive_nba/MyDrive/NBA_Data_Science"),
    Path("/content/nba_drive/MyDrive/NBA_Data_Science"),
]

if _data_override:
    _project_dir = Path(_data_override).expanduser()
else:
    _project_dir = next(
        (p for p in _mount_candidates if (p / "nba_database.db").is_file()),
        None,
    )
    if _project_dir is None:
        _project_dir = next((p for p in _mount_candidates if p.is_dir()), None)
    if _project_dir is None:
        # Keep the existing default Colab mount behavior.
        from google.colab import drive
        if not Path("/content/drive/MyDrive").is_dir():
            drive.mount("/content/drive")
        _project_dir = _mount_candidates[0]

BASE_DIR = str(_project_dir)
print("Project data directory:", BASE_DIR)

import pandas as pd
import numpy as np
import plotly.express as px

from datetime import datetime
from zoneinfo import ZoneInfo
from nba_api.stats.static import teams, players
from nba_api.live.nba.endpoints import scoreboard

# Stagioni da analizzare
STAGIONI = ["2024-25", "2025-26", "2026-27"]
STAGIONE_ATTUALE = "2026-27"

# Elenco delle squadre NBA
squadre = pd.DataFrame(teams.get_teams())

print("=== NBA DATA CENTER ===")
print("Stagione attuale:", STAGIONE_ATTUALE)
print("Squadre disponibili:", len(squadre))

display(
    squadre[["full_name", "abbreviation", "id"]]
)

# Recupero scoreboard NBA live
print("\n=== NBA LIVE SCOREBOARD ===")

try:
    dati_live = scoreboard.ScoreBoard().get_dict()

    partite = dati_live.get("scoreboard", {}).get(
        "games", []
    )

    righe = []

    for partita in partite:

        casa = partita.get("homeTeam", {})
        ospite = partita.get("awayTeam", {})

        righe.append({
            "Casa": casa.get("teamTricode"),
            "Trasferta": ospite.get("teamTricode"),
            "Punti Casa": casa.get("score"),
            "Punti Trasferta": ospite.get("score"),
            "Stato": partita.get("gameStatusText")
        })

    df_partite = pd.DataFrame(righe)

    if df_partite.empty:
        print("Nessuna partita nello scoreboard corrente.")
    else:
        display(df_partite)

except Exception as e:
    print("Scoreboard non disponibile:", str(e))

print("\nConfigurazione completata!")


In [ ]:
# Historical game collection


# ==========================================================
# NBA DATA SCIENCE — STEP 2.3
# ESPN GAME DATABASE BUILDER
# ==========================================================

import os
import time
import sqlite3
import requests
import pandas as pd

from datetime import date, timedelta
from google.colab import drive
from IPython.display import display

# 1. GOOGLE DRIVE

# The setup cell mounts Drive and resolves the data directory.
CARTELLA = BASE_DIR
os.makedirs(CARTELLA, exist_ok=True)

DB_PATH = os.path.join(CARTELLA, "nba_database.db")

# 2. CONFIGURAZIONE

URL = (
    "https://site.api.espn.com/apis/site/v2/"
    "sports/basketball/nba/scoreboard"
)

PERIODI = {
    "2025-26": ("2025-10-21", "2026-04-12"),
    "2026-27": ("2026-10-20", "2026-11-23")
}

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0"
})

conn = sqlite3.connect(DB_PATH)

# 3. TABELLE DATABASE

conn.execute("""
CREATE TABLE IF NOT EXISTS espn_games (
    game_id TEXT PRIMARY KEY,
    season TEXT,
    game_date TEXT,
    home_team TEXT,
    away_team TEXT,
    home_id TEXT,
    away_id TEXT,
    home_score INTEGER,
    away_score INTEGER,
    status TEXT,
    completed INTEGER,
    updated_at TEXT
)
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS espn_download_log (
    season TEXT,
    game_date TEXT,
    games_found INTEGER,
    downloaded_at TEXT,
    PRIMARY KEY (season, game_date)
)
""")

conn.commit()

# 4. FUNZIONE DOWNLOAD

def scarica_giornata(giorno):

    params = {
        "dates": giorno.strftime("%Y%m%d"),
        "limit": 100
    }

    for tentativo in range(3):

        try:
            risposta = session.get(
                URL,
                params=params,
                timeout=20
            )

            risposta.raise_for_status()
            return risposta.json().get("events", [])

        except Exception as e:
            print("Errore:", giorno, str(e)[:100])
            time.sleep(2 * (tentativo + 1))

    return None

# 5. ESTRAZIONE PARTITE

def estrai_partita(evento, stagione):

    competizioni = evento.get("competitions", [])

    if not competizioni:
        return None

    competizione = competizioni[0]
    squadre = competizione.get("competitors", [])

    casa = next(
        (s for s in squadre if s.get("homeAway") == "home"),
        None
    )

    ospite = next(
        (s for s in squadre if s.get("homeAway") == "away"),
        None
    )

    if casa is None or ospite is None:
        return None

    stato = evento.get("status", {}).get("type", {})
    completata = int(stato.get("completed", False))

    def punteggio(s):
        valore = s.get("score")
        try:
            return int(valore)
        except (ValueError, TypeError):
            return None

    return (
        str(evento["id"]),
        stagione,
        evento.get("date"),
        casa.get("team", {}).get("displayName"),
        ospite.get("team", {}).get("displayName"),
        str(casa.get("team", {}).get("id")),
        str(ospite.get("team", {}).get("id")),
        punteggio(casa),
        punteggio(ospite),
        stato.get("name"),
        completata,
        pd.Timestamp.now(tz="UTC").isoformat()
    )

# 6. DOWNLOAD INCREMENTALE

for stagione, (inizio, fine) in PERIODI.items():

    giorno = date.fromisoformat(inizio)
    ultimo = date.fromisoformat(fine)

    print("\nSTAGIONE:", stagione)

    while giorno <= ultimo:

        data_str = giorno.isoformat()

        gia_scaricato = conn.execute(
            """
            SELECT 1 FROM espn_download_log
            WHERE season=? AND game_date=?
            """,
            (stagione, data_str)
        ).fetchone()

        # La stagione corrente viene aggiornata
        if gia_scaricato and stagione != "2026-27":
            giorno += timedelta(days=1)
            continue

        eventi = scarica_giornata(giorno)

        if eventi is None:
            print("Giornata non scaricata:", data_str)
            giorno += timedelta(days=1)
            continue

        righe = []

        for evento in eventi:
            riga = estrai_partita(evento, stagione)
            if riga is not None:
                righe.append(riga)

        with conn:
            conn.executemany(
                """
                INSERT OR REPLACE INTO espn_games
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                """,
                righe
            )

            conn.execute(
                """
                INSERT OR REPLACE INTO espn_download_log
                VALUES (?, ?, ?, ?)
                """,
                (
                    stagione,
                    data_str,
                    len(righe),
                    pd.Timestamp.now(tz="UTC").isoformat()
                )
            )

        print(
            data_str,
            "| Partite:",
            len(righe)
        )

        time.sleep(0.35)
        giorno += timedelta(days=1)

# 7. RIEPILOGO DATABASE

riepilogo = pd.read_sql_query(
    """
    SELECT
        season AS Stagione,
        COUNT(*) AS Partite,
        SUM(completed) AS Completate
    FROM espn_games
    GROUP BY season
    ORDER BY season
    """,
    conn
)

print("\n=== DATABASE NBA ===")
display(riepilogo)

# 8. ULTIME 15 PARTITE

ultime = pd.read_sql_query(
    """
    SELECT
        game_date,
        away_team,
        away_score,
        home_team,
        home_score,
        status
    FROM espn_games
    ORDER BY game_date DESC
    LIMIT 15
    """,
    conn
)

print("\n=== ULTIME PARTITE ===")
display(ultime)

conn.close()

print("\nDATABASE SALVATO:", DB_PATH)
print("STEP 2.3 COMPLETATO!")


In [ ]:
# Player boxscore collection


# ============================================================
# NBA DATA SCIENCE — STEP 3
# ESPN PLAYER BOX SCORE DATABASE
# ============================================================

import os
import time
import sqlite3
import requests
import pandas as pd

from google.colab import drive
from IPython.display import display

# ------------------------------------------------------------
# 1. GOOGLE DRIVE
# ------------------------------------------------------------

# Drive and BASE_DIR are initialized in the setup cell.

DB_PATH = os.path.join(BASE_DIR, "nba_database.db")

if not os.path.isfile(DB_PATH):
    raise FileNotFoundError(
        "Database non trovato. Esegui prima lo STEP 2.3."
    )

conn = sqlite3.connect(DB_PATH)

# ------------------------------------------------------------
# 2. CONFIGURAZIONE
# ------------------------------------------------------------

URL = (
    "https://site.api.espn.com/apis/site/v2/"
    "sports/basketball/nba/summary"
)

# Prima prova con 10 partite.
# Dopo il test cambia a None per scaricare tutte le
# partite concluse non ancora elaborate.
LIMITE_PARTITE = None

PAUSA_API = 1.2
TENTATIVI = 3

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
})

# ------------------------------------------------------------
# 3. CREAZIONE TABELLE
# ------------------------------------------------------------

conn.execute("""
CREATE TABLE IF NOT EXISTS espn_player_boxscores (
    game_id TEXT NOT NULL,
    season TEXT,
    game_date TEXT,
    team_id TEXT NOT NULL,
    team_name TEXT,
    player_id TEXT NOT NULL,
    player_name TEXT,
    starter INTEGER,
    did_not_play INTEGER,
    minutes REAL,
    pts REAL,
    reb REAL,
    ast REAL,
    stl REAL,
    blk REAL,
    tov REAL,
    oreb REAL,
    dreb REAL,
    pf REAL,
    fgm REAL,
    fga REAL,
    fg3m REAL,
    fg3a REAL,
    ftm REAL,
    fta REAL,
    plus_minus REAL,
    PRIMARY KEY (game_id, team_id, player_id)
)
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS espn_boxscore_log (
    game_id TEXT PRIMARY KEY,
    downloaded_at TEXT,
    players_found INTEGER
)
""")

conn.commit()

# ------------------------------------------------------------
# 4. FUNZIONI DI CONVERSIONE
# ------------------------------------------------------------

def numero(valore):

    if valore is None:
        return None

    testo = str(valore).strip()

    if testo in ("", "-", "--", "N/A", "DNP"):
        return None

    try:
        return float(testo.replace(",", ""))
    except (ValueError, TypeError):
        return None


def minuti(valore):

    if valore is None:
        return None

    testo = str(valore).strip()

    if ":" in testo:
        try:
            mm, ss = testo.split(":", 1)
            return int(mm) + int(ss) / 60
        except ValueError:
            return None

    return numero(testo)


def tiri(valore):

    if valore is None:
        return None, None

    testo = str(valore).strip()

    if "-" not in testo:
        return None, None

    try:
        fatti, tentati = testo.split("-", 1)
        return float(fatti), float(tentati)
    except ValueError:
        return None, None

# ------------------------------------------------------------
# 5. DOWNLOAD BOX SCORE
# ------------------------------------------------------------

def scarica_boxscore(game_id):

    for tentativo in range(1, TENTATIVI + 1):

        try:
            risposta = session.get(
                URL,
                params={"event": game_id},
                timeout=25
            )

            risposta.raise_for_status()

            dati = risposta.json()

            if not isinstance(dati, dict):
                raise ValueError("Risposta JSON non valida")

            return dati

        except Exception as errore:

            print(
                f"Errore {game_id} "
                f"({tentativo}/{TENTATIVI}): "
                f"{str(errore)[:100]}"
            )

            time.sleep(2 * tentativo)

    return None

# ------------------------------------------------------------
# 6. ESTRAZIONE DELLE STATISTICHE
# ------------------------------------------------------------

def estrai_giocatori(dati, game_id, stagione, data):

    boxscore = dati.get("boxscore", {})
    gruppi = boxscore.get("players", [])

    righe = []

    for gruppo in gruppi:

        team = gruppo.get("team", {})

        team_id = str(team.get("id", ""))
        team_name = team.get("displayName", "")

        for sezione in gruppo.get("statistics", []):

            nomi = sezione.get("names", [])
            atleti = sezione.get("athletes", [])

            for elemento in atleti:

                atleta = elemento.get("athlete", {})

                player_id = atleta.get("id")

                if player_id is None:
                    continue

                player_name = atleta.get(
                    "displayName",
                    atleta.get("fullName", "")
                )

                valori = elemento.get("stats", [])

                statistiche = dict(zip(nomi, valori))

                # ESPN può usare nomi leggermente diversi.
                def leggi(*chiavi):
                    for chiave in chiavi:
                        if chiave in statistiche:
                            return statistiche[chiave]
                    return None

                fg = leggi("FG", "fieldGoals")
                triple = leggi("3PT", "3P", "threePointFieldGoals")
                liberi = leggi("FT", "freeThrows")

                fgm, fga = tiri(fg)
                fg3m, fg3a = tiri(triple)
                ftm, fta = tiri(liberi)

                # DNP = giocatore che non ha disputato
                # minuti nella partita.
                dnp = bool(
                    elemento.get("didNotPlay", False)
                )

                riga = (
                    str(game_id),
                    stagione,
                    data,
                    team_id,
                    team_name,
                    str(player_id),
                    player_name,
                    int(bool(elemento.get("starter", False))),
                    int(dnp),
                    minuti(leggi("MIN", "minutes")),
                    numero(leggi("PTS", "points")),
                    numero(leggi("REB", "rebounds")),
                    numero(leggi("AST", "assists")),
                    numero(leggi("STL", "steals")),
                    numero(leggi("BLK", "blocks")),
                    numero(leggi("TO", "TOV", "turnovers")),
                    numero(leggi("OREB", "offensiveRebounds")),
                    numero(leggi("DREB", "defensiveRebounds")),
                    numero(leggi("PF", "fouls")),
                    fgm,
                    fga,
                    fg3m,
                    fg3a,
                    ftm,
                    fta,
                    numero(leggi("+/-", "plusMinus"))
                )

                righe.append(riga)

    return righe

# ------------------------------------------------------------
# 7. SELEZIONE PARTITE DA SCARICARE
# ------------------------------------------------------------

query = """
SELECT
    g.game_id,
    g.season,
    g.game_date
FROM espn_games g
LEFT JOIN espn_boxscore_log l
    ON g.game_id = l.game_id
WHERE g.completed = 1
  AND l.game_id IS NULL
ORDER BY g.game_date DESC
"""

partite = pd.read_sql_query(query, conn)

print("Partite ancora da elaborare:", len(partite))

if LIMITE_PARTITE is not None:
    partite = partite.head(LIMITE_PARTITE)

print("Partite selezionate:", len(partite))

# ------------------------------------------------------------
# 8. DOWNLOAD INCREMENTALE
# ------------------------------------------------------------

successi = 0
fallimenti = 0
giocatori_totali = 0

for indice, partita in enumerate(
    partite.itertuples(index=False),
    start=1
):

    game_id = str(partita.game_id)

    dati = scarica_boxscore(game_id)

    if dati is None:
        fallimenti += 1
        continue

    righe = estrai_giocatori(
        dati,
        game_id,
        partita.season,
        partita.game_date
    )

    # Non segnare come completato un box score vuoto.
    if not righe:
        print("Box score vuoto:", game_id)
        fallimenti += 1
        continue

    # Controllo minimo sulla struttura dei dati:
    # almeno una prestazione con punti e minuti.
    valide = [
        r for r in righe
        if r[9] is not None and r[10] is not None
    ]

    if not valide:
        print("Statistiche non valide:", game_id)
        fallimenti += 1
        continue

    with conn:

        conn.executemany(
            """
            INSERT OR REPLACE INTO espn_player_boxscores
            VALUES (
                ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
                ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
                ?, ?, ?, ?, ?, ?
            )
            """,
            righe
        )

        conn.execute(
            """
            INSERT OR REPLACE INTO espn_boxscore_log
            VALUES (?, ?, ?)
            """,
            (
                game_id,
                pd.Timestamp.now(tz="UTC").isoformat(),
                len(righe)
            )
        )

    successi += 1
    giocatori_totali += len(righe)

    print(
        f"[{indice}/{len(partite)}] "
        f"Partita {game_id} | "
        f"Giocatori: {len(righe)}"
    )

    time.sleep(PAUSA_API)

# ------------------------------------------------------------
# 9. CONTROLLO DATABASE
# ------------------------------------------------------------

print("\n========== STEP 3 SUMMARY ==========")

print("Partite scaricate:", successi)
print("Partite fallite:", fallimenti)
print("Righe giocatore scaricate:", giocatori_totali)

riepilogo = pd.read_sql_query(
    """
    SELECT
        season AS Stagione,
        COUNT(DISTINCT game_id) AS Partite,
        COUNT(*) AS Righe_Giocatori,
        COUNT(DISTINCT player_id) AS Giocatori_Unici
    FROM espn_player_boxscores
    GROUP BY season
    ORDER BY season
    """,
    conn
)

print("\n=== RIEPILOGO DATABASE ===")
print(riepilogo.to_string(index=False))

# ------------------------------------------------------------
# 10. TOP SCORER
# ------------------------------------------------------------

top = pd.read_sql_query(
    """
    SELECT
        player_name AS Giocatore,
        COUNT(*) AS GP,
        ROUND(AVG(pts), 1) AS PPG,
        ROUND(AVG(reb), 1) AS RPG,
        ROUND(AVG(ast), 1) AS APG
    FROM espn_player_boxscores
    WHERE season = '2026-27'
      AND did_not_play = 0
      AND minutes > 0
      AND pts IS NOT NULL
    GROUP BY player_id, player_name
    ORDER BY PPG DESC
    LIMIT 20
    """,
    conn
)

print("\n=== TOP SCORER 2026-27 ===")
print(top.to_string(index=False))

# ------------------------------------------------------------
# 11. CHIUSURA
# ------------------------------------------------------------

conn.close()

print("\nDATABASE SALVATO:", DB_PATH)
print("STEP 3 COMPLETATO!")


In [ ]:
# Exploratory analysis


# ============================================================
# NBA DATA SCIENCE & ANALYTICS
# STEP 4 — ADVANCED ANALYTICS
# VERSIONE CORRETTA
# ============================================================

import os
import sqlite3
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from google.colab import drive
from IPython.display import display

# ============================================================
# 1. COLLEGAMENTO GOOGLE DRIVE
# ============================================================

# Drive and BASE_DIR are initialized in the setup cell.

DB_PATH = os.path.join(BASE_DIR, "nba_database.db")

if not os.path.isfile(DB_PATH):
    raise FileNotFoundError(
        "Database non trovato. Completa prima STEP 2 e STEP 3."
    )

conn = sqlite3.connect(DB_PATH)

# Verifica delle tabelle

tabelle = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table'",
    conn
)["name"].tolist()

for tabella in ["espn_games", "espn_player_boxscores"]:
    if tabella not in tabelle:
        conn.close()
        raise ValueError(f"Tabella mancante: {tabella}")

# ============================================================
# 2. CARICAMENTO DATABASE
# ============================================================

games = pd.read_sql_query(
    "SELECT * FROM espn_games",
    conn
)

box = pd.read_sql_query(
    "SELECT * FROM espn_player_boxscores",
    conn
)

conn.close()

games["game_date"] = pd.to_datetime(
    games["game_date"],
    utc=True,
    errors="coerce"
)

box["game_date"] = pd.to_datetime(
    box["game_date"],
    utc=True,
    errors="coerce"
)

print("================================")
print("NBA DATABASE CARICATO")
print("================================")
print("Partite:", len(games))
print("Prestazioni giocatori:", len(box))

# ============================================================
# 3. CONFIGURAZIONE
# ============================================================

STAGIONE = "2026-27"

GIOCATORE = "Nikola Jokic"

GIOCATORE_1 = "Nikola Jokic"
GIOCATORE_2 = "Shai Gilgeous-Alexander"

MIN_GP = 5

# ============================================================
# 4. FILTRO STAGIONE
# ============================================================

df = box[
    (box["season"] == STAGIONE)
    & (box["did_not_play"] == 0)
    & (box["minutes"] > 0)
].copy()

if df.empty:
    raise ValueError(
        f"Nessuna prestazione disponibile per {STAGIONE}"
    )

print("\nStagione:", STAGIONE)
print("Prestazioni valide:", len(df))

# ============================================================
# 5. PREPARAZIONE STATISTICHE
# ============================================================

colonne_numeriche = [
    "minutes", "pts", "reb", "ast",
    "stl", "blk", "tov", "oreb",
    "dreb", "pf", "fgm", "fga",
    "fg3m", "fg3a", "ftm", "fta",
    "plus_minus"
]

for col in colonne_numeriche:

    if col not in df.columns:
        df[col] = np.nan

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

# True Shooting % per singola partita

den_ts = 2 * (
    df["fga"] + 0.44 * df["fta"]
)

df["TS%"] = np.where(
    den_ts > 0,
    df["pts"] / den_ts * 100,
    np.nan
)

# Effective Field Goal % per singola partita

df["eFG%"] = np.where(
    df["fga"] > 0,
    (
        df["fgm"] + 0.5 * df["fg3m"]
    ) / df["fga"] * 100,
    np.nan
)

# ============================================================
# 6. PLAYER RANKINGS
# ============================================================

ranking = (
    df.groupby("player_id")
    .agg(
        Giocatore=("player_name", "last"),
        GP=("game_id", "nunique"),
        MIN=("minutes", "mean"),
        PPG=("pts", "mean"),
        RPG=("reb", "mean"),
        APG=("ast", "mean"),
        SPG=("stl", "mean"),
        BPG=("blk", "mean"),
        TOPG=("tov", "mean"),
        TPM=("fg3m", "mean"),
        FGM=("fgm", "sum"),
        FGA=("fga", "sum"),
        FG3M=("fg3m", "sum"),
        FG3A=("fg3a", "sum"),
        FTM=("ftm", "sum"),
        FTA=("fta", "sum"),
        PTS_TOTAL=("pts", "sum")
    )
    .reset_index()
)

# Percentuale dal campo

ranking["FG%"] = np.where(
    ranking["FGA"] > 0,
    ranking["FGM"] / ranking["FGA"] * 100,
    np.nan
)

# Percentuale da tre punti

ranking["3P%"] = np.where(
    ranking["FG3A"] > 0,
    ranking["FG3M"] / ranking["FG3A"] * 100,
    np.nan
)

# True Shooting % stagionale

den_ts_ranking = 2 * (
    ranking["FGA"] + 0.44 * ranking["FTA"]
)

ranking["TS%"] = np.where(
    den_ts_ranking > 0,
    ranking["PTS_TOTAL"] / den_ts_ranking * 100,
    np.nan
)

# Effective Field Goal % stagionale

ranking["eFG%"] = np.where(
    ranking["FGA"] > 0,
    (
        ranking["FGM"] + 0.5 * ranking["FG3M"]
    ) / ranking["FGA"] * 100,
    np.nan
)

# Ranking con minimo partite

ranking_validi = ranking[
    ranking["GP"] >= MIN_GP
].copy()

ranking_validi = ranking_validi.sort_values(
    "PPG",
    ascending=False
)

print("\n================================")
print("TOP 20 NBA SCORERS")
print("================================")

colonne_ranking = [
    "Giocatore", "GP", "MIN",
    "PPG", "RPG", "APG",
    "TPM", "FG%", "3P%",
    "TS%", "eFG%"
]

display(
    ranking_validi[
        colonne_ranking
    ].head(20).round(2)
)

# ============================================================
# 7. FUNZIONI ULTIME PARTITE
# ============================================================

def ultime_partite(nome, n=10):

    dati = df[
        df["player_name"].str.casefold()
        == nome.casefold()
    ].sort_values(
        ["game_date", "game_id"],
        ascending=False
    )

    return dati.head(n)


def medie_recenti(nome, n=10):

    dati = ultime_partite(nome, n)

    if dati.empty:
        return None

    return {
        "Giocatore": nome,
        "GP": len(dati),
        "PPG": dati["pts"].mean(),
        "RPG": dati["reb"].mean(),
        "APG": dati["ast"].mean(),
        "3PM": dati["fg3m"].mean(),
        "MIN": dati["minutes"].mean()
    }

# ============================================================
# 8. PLAYER DASHBOARD
# ============================================================

print("\n================================")
print("PLAYER DASHBOARD")
print("================================")

dati_giocatore = df[
    df["player_name"].str.casefold()
    == GIOCATORE.casefold()
].sort_values("game_date")

if not dati_giocatore.empty:

    print("Giocatore:", GIOCATORE)

    for n in [5, 10, 20]:

        stats = medie_recenti(GIOCATORE, n)

        print(
            f"Ultime {n} disponibili "
            f"(GP={stats['GP']}): "
            f"PPG={stats['PPG']:.1f} | "
            f"RPG={stats['RPG']:.1f} | "
            f"APG={stats['APG']:.1f}"
        )

    fig = go.Figure()

    for col, nome in [
        ("pts", "Punti"),
        ("reb", "Rimbalzi"),
        ("ast", "Assist")
    ]:

        fig.add_trace(
            go.Scatter(
                x=dati_giocatore["game_date"],
                y=dati_giocatore[col],
                mode="lines+markers",
                name=nome
            )
        )

    fig.update_layout(
        title=f"{GIOCATORE} — Performance Trend",
        template="plotly_dark",
        height=500,
        xaxis_title="Data",
        yaxis_title="Statistica"
    )

    fig.show()

else:

    print(
        f"{GIOCATORE} non trovato "
        "nel database disponibile."
    )

# ============================================================
# 9. TEAM ANALYTICS
# ============================================================

partite_concluse = games[
    (games["season"] == STAGIONE)
    & (games["completed"] == 1)
    & games["home_score"].notna()
    & games["away_score"].notna()
].copy()

righe_team = []

for _, partita in partite_concluse.iterrows():

    casa = {
        "Squadra": partita["home_team"],
        "Data": partita["game_date"],
        "Punti Fatti": partita["home_score"],
        "Punti Subiti": partita["away_score"],
        "Vittoria": int(
            partita["home_score"] > partita["away_score"]
        )
    }

    ospite = {
        "Squadra": partita["away_team"],
        "Data": partita["game_date"],
        "Punti Fatti": partita["away_score"],
        "Punti Subiti": partita["home_score"],
        "Vittoria": int(
            partita["away_score"] > partita["home_score"]
        )
    }

    righe_team.extend([casa, ospite])

team_games = pd.DataFrame(righe_team)

if team_games.empty:
    raise ValueError(
        "Nessuna partita conclusa disponibile."
    )

team_ranking = (
    team_games.groupby("Squadra")
    .agg(
        GP=("Vittoria", "count"),
        W=("Vittoria", "sum"),
        PPG=("Punti Fatti", "mean"),
        OPP_PPG=("Punti Subiti", "mean")
    )
    .reset_index()
)

team_ranking["L"] = (
    team_ranking["GP"] - team_ranking["W"]
)

team_ranking["WIN%"] = (
    team_ranking["W"] / team_ranking["GP"] * 100
)

team_ranking["DIFF"] = (
    team_ranking["PPG"] - team_ranking["OPP_PPG"]
)

team_ranking = team_ranking.sort_values(
    ["WIN%", "DIFF"],
    ascending=False
)

print("\n================================")
print("NBA TEAM RANKINGS")
print("================================")

display(
    team_ranking[
        [
            "Squadra", "GP", "W", "L",
            "WIN%", "PPG", "OPP_PPG", "DIFF"
        ]
    ].round(2)
)

# ============================================================
# 10. TOP SCORERS CHART
# ============================================================

if not ranking_validi.empty:

    fig = px.bar(
        ranking_validi.head(20),
        x="Giocatore",
        y="PPG",
        color="PPG",
        title=f"NBA {STAGIONE} — Top 20 Scorers",
        template="plotly_dark"
    )

    fig.update_layout(
        height=550,
        xaxis_tickangle=-45
    )

    fig.show()

# ============================================================
# 11. PLAYER COMPARISON
# ============================================================

confronto = ranking[
    ranking["Giocatore"].isin([
        GIOCATORE_1,
        GIOCATORE_2
    ])
][
    ["Giocatore", "GP", "PPG", "RPG", "APG", "TS%"]
].copy()

print("\n================================")
print("PLAYER COMPARISON")
print("================================")

display(confronto.round(2))

if len(confronto) == 2:

    confronto_long = confronto.melt(
        id_vars=["Giocatore", "GP"],
        value_vars=["PPG", "RPG", "APG", "TS%"],
        var_name="Statistica",
        value_name="Valore"
    )

    fig = px.bar(
        confronto_long,
        x="Statistica",
        y="Valore",
        color="Giocatore",
        barmode="group",
        title=(
            f"{GIOCATORE_1} vs "
            f"{GIOCATORE_2}"
        ),
        template="plotly_dark"
    )

    fig.show()

# ============================================================
# 12. EXPORT EXCEL
# ============================================================

OUTPUT = (
    "/content/drive/MyDrive/NBA_Data_Science/"
    "NBA_Advanced_Analytics.xlsx"
)

with pd.ExcelWriter(
    OUTPUT,
    engine="openpyxl"
) as writer:

    ranking.to_excel(
        writer,
        sheet_name="Player Rankings",
        index=False
    )

    team_ranking.to_excel(
        writer,
        sheet_name="Team Rankings",
        index=False
    )

    df.to_excel(
        writer,
        sheet_name="Player Game Logs",
        index=False
    )

    team_games.to_excel(
        writer,
        sheet_name="Team Game Logs",
        index=False
    )

print("\n================================")
print("STEP 4 COMPLETATO!")
print("================================")

print("Stagione analizzata:", STAGIONE)
print("Giocatori analizzati:", len(ranking))
print("Partite squadre:", len(partite_concluse))
print("Excel salvato:", OUTPUT)


In [ ]:
# Team ML prediction


# ============================================================
# NBA DATA SCIENCE & ANALYTICS
# STEP 5 — MACHINE LEARNING WIN PREDICTION
# ============================================================

import os
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from collections import defaultdict, deque

from google.colab import drive

from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

from sklearn.metrics import (
    accuracy_score,
    log_loss,
    brier_score_loss,
    roc_auc_score
)

# ============================================================
# 1. CARICAMENTO DATABASE
# ============================================================

# Drive and BASE_DIR are initialized in the setup cell.
CARTELLA = BASE_DIR
DB_PATH = os.path.join(CARTELLA, "nba_database.db")

if not os.path.isfile(DB_PATH):
    raise FileNotFoundError("Database NBA non trovato.")

conn = sqlite3.connect(DB_PATH)

games = pd.read_sql_query(
    """
    SELECT
        game_id,
        season,
        game_date,
        home_id,
        away_id,
        home_team,
        away_team,
        home_score,
        away_score,
        completed
    FROM espn_games
    """,
    conn
)

conn.close()

# ============================================================
# 2. PULIZIA PARTITE
# ============================================================

games["game_date"] = pd.to_datetime(
    games["game_date"],
    utc=True,
    errors="coerce"
)

games["home_score"] = pd.to_numeric(
    games["home_score"],
    errors="coerce"
)

games["away_score"] = pd.to_numeric(
    games["away_score"],
    errors="coerce"
)

games = games[
    (games["completed"] == 1)
    & games["game_date"].notna()
    & games["home_score"].notna()
    & games["away_score"].notna()
    & games["home_id"].notna()
    & games["away_id"].notna()
    & (games["home_score"] != games["away_score"])
].copy()

games = games.sort_values(
    ["game_date", "game_id"]
).drop_duplicates("game_id")

print("Partite valide:", len(games))

if len(games) < 200:
    raise ValueError(
        "Troppo poche partite per questo esperimento."
    )

# ============================================================
# 3. FEATURE ENGINEERING
# ============================================================

# Manteniamo la cronologia di ogni squadra.
# Le feature vengono calcolate prima di aggiungere
# il risultato della partita corrente.

storici = defaultdict(list)
righe = []

def statistiche_prepartita(storico, data):

    if len(storico) == 0:
        return None

    ultime_5 = storico[-5:]
    ultime_10 = storico[-10:]

    def media(partite, chiave):
        return float(
            np.mean([p[chiave] for p in partite])
        )

    giorni_riposo = (
        data.date() - storico[-1]["data"].date()
    ).days - 1

    return {
        "win5": media(ultime_5, "win"),
        "win10": media(ultime_10, "win"),
        "pf5": media(ultime_5, "pf"),
        "pa5": media(ultime_5, "pa"),
        "diff5": media(ultime_5, "diff"),
        "diff10": media(ultime_10, "diff"),
        "rest": float(
            np.clip(giorni_riposo, 0, 7)
        ),
        "gp": len(storico)
    }

for partita in games.itertuples(index=False):

    home_id = str(partita.home_id)
    away_id = str(partita.away_id)
    data = partita.game_date

    home_hist = storici[home_id]
    away_hist = storici[away_id]

    home = statistiche_prepartita(home_hist, data)
    away = statistiche_prepartita(away_hist, data)

    # Richiediamo almeno 5 partite precedenti
    # per entrambe le squadre.
    if (
        home is not None
        and away is not None
        and home["gp"] >= 5
        and away["gp"] >= 5
    ):

        riga = {
            "game_id": partita.game_id,
            "date": data,
            "season": partita.season,
            "home_team": partita.home_team,
            "away_team": partita.away_team,
            "home_win": int(
                partita.home_score > partita.away_score
            )
        }

        for chiave in [
            "win5", "win10", "pf5", "pa5",
            "diff5", "diff10", "rest", "gp"
        ]:

            riga[f"home_{chiave}"] = home[chiave]
            riga[f"away_{chiave}"] = away[chiave]

            riga[f"delta_{chiave}"] = (
                home[chiave] - away[chiave]
            )

        righe.append(riga)

    # Aggiorniamo gli storici soltanto DOPO
    # aver calcolato le feature della partita.

    hp = float(partita.home_score)
    ap = float(partita.away_score)

    home_hist.append({
        "data": data,
        "win": int(hp > ap),
        "pf": hp,
        "pa": ap,
        "diff": hp - ap
    })

    away_hist.append({
        "data": data,
        "win": int(ap > hp),
        "pf": ap,
        "pa": hp,
        "diff": ap - hp
    })

dataset = pd.DataFrame(righe)

print("Dataset ML:", dataset.shape)
print("Feature engineering completato.")

# ============================================================
# 4. TRAIN / VALIDATION / TEST
# ============================================================

# Suddivisione cronologica:
# 70% training
# 15% validation
# 15% test

dataset = dataset.sort_values(
    ["date", "game_id"]
).reset_index(drop=True)

feature_cols = [
    col for col in dataset.columns
    if col.startswith("delta_")
]

n = len(dataset)

fine_train = int(n * 0.70)
fine_val = int(n * 0.85)

train = dataset.iloc[:fine_train]
validation = dataset.iloc[fine_train:fine_val]
test = dataset.iloc[fine_val:]

X_train = train[feature_cols]
y_train = train["home_win"]

X_val = validation[feature_cols]
y_val = validation["home_win"]

X_test = test[feature_cols]
y_test = test["home_win"]

print("\n=== DATA SPLIT ===")
print("Training:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

print(
    "Test dal:",
    test["date"].min(),
    "al:",
    test["date"].max()
)

# ============================================================
# 5. MODELLI
# ============================================================

modelli = {
    "Baseline": DummyClassifier(
        strategy="prior"
    ),

    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1.0,
            max_iter=2000,
            random_state=42
        )
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        min_samples_leaf=8,
        random_state=42,
        n_jobs=-1
    )
}

# ============================================================
# 6. VALUTAZIONE
# ============================================================

def valuta_modello(y, probabilita):

    probabilita = np.clip(
        np.asarray(probabilita),
        1e-6,
        1 - 1e-6
    )

    predizioni = (
        probabilita >= 0.5
    ).astype(int)

    risultati = {
        "Accuracy": accuracy_score(
            y, predizioni
        ),
        "Log Loss": log_loss(
            y, probabilita,
            labels=[0, 1]
        ),
        "Brier Score": brier_score_loss(
            y, probabilita
        )
    }

    if len(np.unique(y)) == 2:
        risultati["ROC AUC"] = roc_auc_score(
            y, probabilita
        )
    else:
        risultati["ROC AUC"] = np.nan

    return risultati

risultati_val = []
modelli_addestrati = {}

for nome, modello in modelli.items():

    fitted = clone(modello)

    fitted.fit(X_train, y_train)

    probabilita = fitted.predict_proba(
        X_val
    )[:, 1]

    metriche = valuta_modello(
        y_val, probabilita
    )

    risultati_val.append({
        "Modello": nome,
        **metriche
    })

    modelli_addestrati[nome] = fitted

classifica = pd.DataFrame(
    risultati_val
).sort_values(
    "Log Loss"
).reset_index(drop=True)

print("\n=== VALIDATION RESULTS ===")
print(classifica.round(4).to_string(index=False))

# ============================================================
# 7. SELEZIONE MIGLIOR MODELLO
# ============================================================

nome_migliore = classifica.iloc[0]["Modello"]

print("\nMiglior modello:", nome_migliore)

# Addestramento su TRAIN + VALIDATION
train_val = dataset.iloc[:fine_val]

X_train_val = train_val[feature_cols]
y_train_val = train_val["home_win"]

modello_finale = clone(
    modelli[nome_migliore]
)

modello_finale.fit(
    X_train_val,
    y_train_val
)

# ============================================================
# 8. TEST OUT-OF-SAMPLE
# ============================================================

prob_test = modello_finale.predict_proba(
    X_test
)[:, 1]

metriche_test = valuta_modello(
    y_test, prob_test
)

print("\n=== TEST RESULTS ===")

for metrica, valore in metriche_test.items():
    print(f"{metrica}: {valore:.4f}")

risultati_test = test[
    [
        "date", "home_team",
        "away_team", "home_win"
    ]
].copy()

risultati_test["P_HOME_WIN"] = prob_test

risultati_test["P_AWAY_WIN"] = (
    1 - prob_test
)

risultati_test["Predicted_Home_Win"] = (
    prob_test >= 0.5
).astype(int)

print("\n=== ESEMPI PREVISIONI TEST ===")
print(
    risultati_test.tail(15).to_string(index=False)
)

# ============================================================
# 9. GRAFICI VALUTAZIONE
# ============================================================

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    classifica["Modello"],
    classifica["Log Loss"]
)

ax.set_title("NBA — Validation Log Loss")
ax.set_ylabel("Log Loss (minore è meglio)")

plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# ============================================================
# 10. SALVATAGGIO MODELLO
# ============================================================

MODEL_PATH = os.path.join(
    CARTELLA,
    "nba_win_prediction_model.joblib"
)

DATASET_PATH = os.path.join(
    CARTELLA,
    "nba_ml_features.csv"
)

TEST_PATH = os.path.join(
    CARTELLA,
    "nba_test_predictions.csv"
)

joblib.dump(
    {
        "model": modello_finale,
        "features": feature_cols,
        "model_name": nome_migliore,
        "test_metrics": metriche_test
    },
    MODEL_PATH
)

dataset.to_csv(
    DATASET_PATH,
    index=False
)

risultati_test.to_csv(
    TEST_PATH,
    index=False
)

print("\n================================")
print("STEP 5 COMPLETATO!")
print("================================")

print("Modello:", nome_migliore)
print("File modello:", MODEL_PATH)
print("Dataset ML:", DATASET_PATH)
print("Previsioni test:", TEST_PATH)


In [ ]:
# Upcoming game prediction


# ============================================================
# NBA DATA SCIENCE & ANALYTICS
# STEP 6 — FORECAST PROSSIME PARTITE NBA
# ============================================================

import os
import sqlite3
import requests
import joblib
import numpy as np
import pandas as pd
import plotly.express as px

from collections import defaultdict
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from google.colab import drive
from IPython.display import display

# ============================================================
# 1. CONFIGURAZIONE
# ============================================================

# Drive and BASE_DIR are initialized in the setup cell.
CARTELLA = BASE_DIR

DB_PATH = os.path.join(
    CARTELLA, "nba_database.db"
)

MODEL_PATH = os.path.join(
    CARTELLA, "nba_win_prediction_model.joblib"
)

GIORNI_FORECAST = 3

if not os.path.isfile(DB_PATH):
    raise FileNotFoundError("Database NBA mancante.")

if not os.path.isfile(MODEL_PATH):
    raise FileNotFoundError(
        "Modello ML mancante. Completa lo STEP 5."
    )

# Caricare file joblib soltanto da fonti attendibili.
pacchetto = joblib.load(MODEL_PATH)

modello = pacchetto["model"]
feature_cols = pacchetto["features"]

print("Modello caricato:", pacchetto["model_name"])
print("Feature richieste:", len(feature_cols))

# ============================================================
# 2. CARICAMENTO RISULTATI STORICI
# ============================================================

conn = sqlite3.connect(DB_PATH)

games = pd.read_sql_query("""
    SELECT
        game_id,
        season,
        game_date,
        home_id,
        away_id,
        home_team,
        away_team,
        home_score,
        away_score,
        completed
    FROM espn_games
""", conn)

conn.close()

games["game_date"] = pd.to_datetime(
    games["game_date"],
    utc=True,
    errors="coerce"
)

for col in ["home_score", "away_score", "completed"]:
    games[col] = pd.to_numeric(
        games[col],
        errors="coerce"
    )

games = games[
    (games["completed"] == 1)
    & games["game_date"].notna()
    & games["home_score"].notna()
    & games["away_score"].notna()
    & games["home_id"].notna()
    & games["away_id"].notna()
    & (games["home_score"] != games["away_score"])
].copy()

games = games.sort_values(
    ["game_date", "game_id"]
).drop_duplicates("game_id")

print("Partite storiche:", len(games))

# ============================================================
# 3. DOWNLOAD CALENDARIO ESPN
# ============================================================

URL = (
    "https://site.api.espn.com/apis/site/v2/"
    "sports/basketball/nba/scoreboard"
)

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
})

oggi_ny = datetime.now(
    ZoneInfo("America/New_York")
).date()

eventi_futuri = {}

for offset in range(GIORNI_FORECAST + 1):

    giorno = oggi_ny + timedelta(days=offset)

    try:
        risposta = session.get(
            URL,
            params={
                "dates": giorno.strftime("%Y%m%d"),
                "limit": 100
            },
            timeout=20
        )

        risposta.raise_for_status()

        eventi = risposta.json().get("events", [])

        for evento in eventi:
            eventi_futuri[str(evento["id"])] = evento

        print(
            giorno,
            "| Eventi:",
            len(eventi)
        )

    except Exception as errore:
        print(
            "Errore calendario:",
            giorno,
            str(errore)[:120]
        )

# ============================================================
# 4. ESTRAZIONE PARTITE PROGRAMMATE
# ============================================================

partite_future = []

for evento in eventi_futuri.values():

    stato = evento.get("status", {}).get("type", {})

    # Solo partite non ancora iniziate
    if stato.get("name") != "STATUS_SCHEDULED":
        continue

    competizioni = evento.get("competitions", [])

    if not competizioni:
        continue

    squadre = competizioni[0].get("competitors", [])

    casa = next(
        (
            s for s in squadre
            if s.get("homeAway") == "home"
        ),
        None
    )

    ospite = next(
        (
            s for s in squadre
            if s.get("homeAway") == "away"
        ),
        None
    )

    if casa is None or ospite is None:
        continue

    data = pd.to_datetime(
        evento.get("date"),
        utc=True,
        errors="coerce"
    )

    if pd.isna(data):
        continue

    # Evita incontri il cui orario è già passato
    if data <= pd.Timestamp.now(tz="UTC"):
        continue

    partite_future.append({
        "game_id": str(evento["id"]),
        "date": data,
        "home_id": str(casa["team"]["id"]),
        "away_id": str(ospite["team"]["id"]),
        "home_team": casa["team"]["displayName"],
        "away_team": ospite["team"]["displayName"]
    })

future = pd.DataFrame(partite_future)

if future.empty:
    raise ValueError(
        "Nessuna partita futura trovata. "
        "Controlla calendario ESPN e intervallo."
    )

future = future.sort_values("date").reset_index(drop=True)

print("\nPARTITE FUTURE TROVATE:", len(future))

# ============================================================
# 5. STORICO DELLE SQUADRE
# ============================================================

storici = defaultdict(list)

# Utilizziamo solo gare già terminate prima del momento
# in cui vengono generate le previsioni.
ora_utc = pd.Timestamp.now(tz="UTC")

games = games[games["game_date"] < ora_utc].copy()

for partita in games.itertuples(index=False):

    data = partita.game_date

    hp = float(partita.home_score)
    ap = float(partita.away_score)

    storici[str(partita.home_id)].append({
        "data": data,
        "win": int(hp > ap),
        "pf": hp,
        "pa": ap,
        "diff": hp - ap
    })

    storici[str(partita.away_id)].append({
        "data": data,
        "win": int(ap > hp),
        "pf": ap,
        "pa": hp,
        "diff": ap - hp
    })

# ============================================================
# 6. FEATURE ENGINEERING
# Stesse definizioni dello STEP 5
# ============================================================

def statistiche_prepartita(storico, data):

    # Escludiamo sempre incontri non precedenti
    storico = [
        p for p in storico
        if p["data"] < data
    ]

    if len(storico) < 5:
        return None

    ultime_5 = storico[-5:]
    ultime_10 = storico[-10:]

    def media(partite, chiave):
        return float(
            np.mean([p[chiave] for p in partite])
        )

    giorni_riposo = (
        data.date() - storico[-1]["data"].date()
    ).days - 1

    return {
        "win5": media(ultime_5, "win"),
        "win10": media(ultime_10, "win"),
        "pf5": media(ultime_5, "pf"),
        "pa5": media(ultime_5, "pa"),
        "diff5": media(ultime_5, "diff"),
        "diff10": media(ultime_10, "diff"),
        "rest": float(np.clip(giorni_riposo, 0, 7)),
        "gp": len(storico)
    }

# ============================================================
# 7. COSTRUZIONE FEATURES FUTURE
# ============================================================

righe_ml = []
partite_escluse = []

for partita in future.itertuples(index=False):

    home = statistiche_prepartita(
        storici[str(partita.home_id)],
        partita.date
    )

    away = statistiche_prepartita(
        storici[str(partita.away_id)],
        partita.date
    )

    if home is None or away is None:
        partite_escluse.append(
            f"{partita.away_team} @ {partita.home_team}"
        )
        continue

    riga = {
        "game_id": partita.game_id,
        "date": partita.date,
        "home_team": partita.home_team,
        "away_team": partita.away_team
    }

    for chiave in [
        "win5", "win10", "pf5", "pa5",
        "diff5", "diff10", "rest", "gp"
    ]:
        riga[f"delta_{chiave}"] = (
            home[chiave] - away[chiave]
        )

    righe_ml.append(riga)

forecast = pd.DataFrame(righe_ml)

if forecast.empty:
    raise ValueError(
        "Nessuna partita con storico sufficiente."
    )

# Controllo delle feature richieste dal modello
mancanti = [
    col for col in feature_cols
    if col not in forecast.columns
]

if mancanti:
    raise ValueError(
        f"Feature mancanti: {mancanti}"
    )

X_future = forecast[feature_cols].copy()

# ============================================================
# 8. PREVISIONI MACHINE LEARNING
# ============================================================

prob_home = modello.predict_proba(
    X_future
)[:, 1]

forecast["P_HOME_WIN"] = prob_home
forecast["P_AWAY_WIN"] = 1 - prob_home

forecast["P_HOME_%"] = (
    forecast["P_HOME_WIN"] * 100
)

forecast["P_AWAY_%"] = (
    forecast["P_AWAY_WIN"] * 100
)

forecast["Favorita"] = np.where(
    forecast["P_HOME_WIN"] >= 0.5,
    forecast["home_team"],
    forecast["away_team"]
)

forecast["Confidenza_%"] = (
    forecast[
        ["P_HOME_%", "P_AWAY_%"]
    ].max(axis=1)
)

forecast["Data_Italia"] = (
    forecast["date"]
    .dt.tz_convert("Europe/Rome")
    .dt.strftime("%d/%m/%Y %H:%M")
)

# ============================================================
# 9. TABELLA FORECAST
# ============================================================

tabella = forecast[
    [
        "Data_Italia",
        "away_team",
        "home_team",
        "P_AWAY_%",
        "P_HOME_%",
        "Favorita",
        "Confidenza_%"
    ]
].copy()

tabella.columns = [
    "Data Italia",
    "Trasferta",
    "Casa",
    "Vittoria Trasferta %",
    "Vittoria Casa %",
    "Favorita",
    "Probabilità Favorita %"
]

tabella = tabella.round(1)

print("\n====================================")
print("NBA FORECAST — PROSSIME PARTITE")
print("====================================")

display(tabella)

# ============================================================
# 10. DASHBOARD INTERATTIVA
# ============================================================

grafico = forecast.copy()

grafico["Partita"] = (
    grafico["away_team"]
    + " @ "
    + grafico["home_team"]
)

fig = px.bar(
    grafico,
    x="Partita",
    y=["P_HOME_%", "P_AWAY_%"],
    barmode="group",
    title="NBA — Probabilità di vittoria",
    template="plotly_dark",
    labels={
        "value": "Probabilità (%)",
        "variable": "Previsione"
    }
)

fig.update_layout(
    height=550,
    xaxis_tickangle=-35,
    yaxis_range=[0, 100]
)

fig.show()

# ============================================================
# 11. ESPORTAZIONE
# ============================================================

CSV_PATH = os.path.join(
    CARTELLA,
    "NBA_Forecast_Prossime_Partite.csv"
)

EXCEL_PATH = os.path.join(
    CARTELLA,
    "NBA_Forecast_Prossime_Partite.xlsx"
)

tabella.to_csv(
    CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

with pd.ExcelWriter(
    EXCEL_PATH,
    engine="openpyxl"
) as writer:

    tabella.to_excel(
        writer,
        sheet_name="NBA Forecast",
        index=False
    )

print("\n====================================")
print("STEP 6 COMPLETATO!")
print("====================================")

print("Partite previste:", len(forecast))
print("Partite escluse:", len(partite_escluse))
print("CSV:", CSV_PATH)
print("Excel:", EXCEL_PATH)

if partite_escluse:
    print("\nPartite senza storico sufficiente:")
    for nome in partite_escluse:
        print("-", nome)


In [ ]:
# Incremental database update


# ============================================================
# STEP 12 — NBA DATABASE UPDATER
# ============================================================

import os
import time
import sqlite3
import requests
import pandas as pd

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from tqdm.auto import tqdm

BASE_DIR = globals().get("BASE_DIR") or os.environ.get("NBA_DATA_DIR") or "/content/drive/MyDrive/NBA_Data_Science"
DB_PATH = os.path.join(BASE_DIR, "nba_database.db")

if not os.path.isfile(DB_PATH):
    raise FileNotFoundError(DB_PATH)

API = (
    "https://site.api.espn.com/apis/"
    "site/v2/sports/basketball/nba"
)

session = requests.Session()

session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
})

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA busy_timeout = 10000")

def fetch(endpoint, params):

    for attempt in range(3):

        try:

            response = session.get(
                API + endpoint,
                params=params,
                timeout=30
            )

            response.raise_for_status()
            return response.json()

        except (requests.RequestException, ValueError):

            time.sleep(2 * (attempt + 1))

    return None


def number(value):

    try:
        return float(value)
    except (ValueError, TypeError):
        return None


def minutes(value):

    if ":" in str(value):

        try:
            m, s = str(value).split(":")
            return float(m) + float(s) / 60
        except ValueError:
            return None

    return number(value)


def split_stat(value):

    try:
        a, b = str(value).split("-")
        return float(a), float(b)
    except (ValueError, TypeError):
        return None, None


def season_from_date(value):

    d = pd.Timestamp(value).tz_convert(
        "America/New_York"
    )

    y = d.year if d.month >= 7 else d.year - 1

    return f"{y}-{str(y + 1)[-2:]}"


# ============================================================
# CONTROLLO SCHEMA
# ============================================================

game_columns = {
    row[1]
    for row in conn.execute(
        "PRAGMA table_info(espn_games)"
    )
}

for column in ["home_id", "away_id"]:

    if column not in game_columns:

        conn.execute(
            f"ALTER TABLE espn_games "
            f"ADD COLUMN {column} TEXT"
        )

conn.commit()

# ============================================================
# AGGIORNAMENTO PARTITE
# ============================================================

today = datetime.now(
    ZoneInfo("America/New_York")
).date()

events = {}
failed_dates = []

for offset in tqdm(
    range(-7, 4),
    desc="Calendario NBA"
):

    day = today + timedelta(days=offset)

    result = fetch(
        "/scoreboard",
        {
            "dates": day.strftime("%Y%m%d"),
            "limit": 100
        }
    )

    if result is None:
        failed_dates.append(str(day))
        continue

    for event in result.get("events", []):

        events[str(event["id"])] = event

    time.sleep(0.2)

updated = 0

for gid, event in events.items():

    competitions = event.get("competitions", [])

    if not competitions:
        continue

    teams = competitions[0].get("competitors", [])

    home = next(
        (t for t in teams if t["homeAway"] == "home"),
        None
    )

    away = next(
        (t for t in teams if t["homeAway"] == "away"),
        None
    )

    if not home or not away:
        continue

    status = event.get("status", {}).get("type", {})
    completed = int(status.get("completed", False))

    previous = conn.execute(
        "SELECT completed FROM espn_games WHERE game_id=?",
        (gid,)
    ).fetchone()

    if previous and previous[0] == 1 and not completed:
        continue

    values = (
        season_from_date(event["date"]),
        event["date"],
        str(home["team"]["id"]),
        str(away["team"]["id"]),
        home["team"]["displayName"],
        away["team"]["displayName"],
        number(home.get("score")),
        number(away.get("score")),
        completed,
        status.get("name", "UNKNOWN")
    )

    with conn:

        if previous:

            conn.execute("""
                UPDATE espn_games SET
                    season=?,
                    game_date=?,
                    home_id=?,
                    away_id=?,
                    home_team=?,
                    away_team=?,
                    home_score=?,
                    away_score=?,
                    completed=?,
                    status=?
                WHERE game_id=?
            """, values + (gid,))

        else:

            conn.execute("""
                INSERT INTO espn_games (
                    season, game_date,
                    home_id, away_id,
                    home_team, away_team,
                    home_score, away_score,
                    completed, status, game_id
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, values + (gid,))

    updated += 1

# ============================================================
# INDIVIDUAZIONE BOXSCORE MANCANTI
# ============================================================

missing_games = conn.execute("""
    SELECT g.game_id, g.season, g.game_date
    FROM espn_games g
    WHERE g.completed = 1
    AND (
        SELECT COUNT(DISTINCT b.team_id)
        FROM espn_player_boxscores b
        WHERE b.game_id = g.game_id
        AND b.minutes > 0
        AND b.pts IS NOT NULL
    ) < 2
    ORDER BY g.game_date
""").fetchall()

print("Box score mancanti:", len(missing_games))

BOX_COLS = [
    "game_id", "season", "game_date",
    "team_id", "team_name",
    "player_id", "player_name",
    "starter", "did_not_play",
    "minutes", "pts", "reb", "ast",
    "stl", "blk", "tov", "oreb",
    "dreb", "pf", "fgm", "fga",
    "fg3m", "fg3a", "ftm", "fta",
    "plus_minus"
]

box_schema = {
    row[1]
    for row in conn.execute(
        "PRAGMA table_info(espn_player_boxscores)"
    )
}

missing_cols = set(BOX_COLS) - box_schema

if missing_cols:
    raise ValueError(
        f"Schema boxscore incompatibile: {missing_cols}"
    )

# ============================================================
# DOWNLOAD STATISTICHE
# ============================================================

downloaded = 0
failed = []

for gid, season, game_date in tqdm(
    missing_games,
    desc="Player Box Scores"
):

    result = fetch(
        "/summary",
        {"event": gid}
    )

    if not result:
        failed.append(gid)
        continue

    rows = []
    valid_teams = set()

    for team_data in result.get(
        "boxscore", {}
    ).get("players", []):

        team = team_data.get("team", {})
        team_id = str(team.get("id", ""))

        for group in team_data.get("statistics", []):

            labels = group.get("labels") or group.get("names", [])

            labels = [
                str(x).upper()
                for x in labels
            ]

            for athlete in group.get("athletes", []):

                player = athlete.get("athlete", {})
                player_id = str(player.get("id", ""))

                if not player_id or not team_id:
                    continue

                stats = dict(zip(
                    labels,
                    athlete.get("stats", [])
                ))

                fgm, fga = split_stat(stats.get("FG"))
                fg3m, fg3a = split_stat(stats.get("3PT"))
                ftm, fta = split_stat(stats.get("FT"))

                mins = minutes(stats.get("MIN"))
                pts = number(stats.get("PTS"))

                if mins is not None and mins > 0 and pts is not None:
                    valid_teams.add(team_id)

                rows.append((
                    str(gid),
                    season,
                    game_date,
                    team_id,
                    team.get("displayName", ""),
                    player_id,
                    player.get("displayName", ""),
                    int(bool(athlete.get("starter"))),
                    int(bool(athlete.get("didNotPlay"))),
                    mins,
                    pts,
                    number(stats.get("REB")),
                    number(stats.get("AST")),
                    number(stats.get("STL")),
                    number(stats.get("BLK")),
                    number(stats.get("TO", stats.get("TOV"))),
                    number(stats.get("OREB")),
                    number(stats.get("DREB")),
                    number(stats.get("PF")),
                    fgm, fga,
                    fg3m, fg3a,
                    ftm, fta,
                    number(stats.get("+/-"))
                ))

    if len(valid_teams) != 2:
        failed.append(gid)
        continue

    sql = (
        "INSERT INTO espn_player_boxscores ("
        + ",".join(BOX_COLS)
        + ") VALUES ("
        + ",".join(["?"] * len(BOX_COLS))
        + ")"
    )

    try:

        with conn:

            conn.execute("""
                DELETE FROM espn_player_boxscores
                WHERE game_id = ?
            """, (gid,))

            conn.executemany(sql, rows)

        downloaded += 1

    except sqlite3.Error as e:

        failed.append(gid)
        print("Errore:", gid, e)

    time.sleep(0.25)

# ============================================================
# REPORT
# ============================================================

total_games = conn.execute(
    "SELECT COUNT(*) FROM espn_games"
).fetchone()[0]

total_box = conn.execute(
    "SELECT COUNT(*) FROM espn_player_boxscores"
).fetchone()[0]

conn.close()
session.close()

report = pd.DataFrame([{
    "Timestamp": pd.Timestamp.now(tz="UTC").isoformat(),
    "Partite database": total_games,
    "Partite aggiornate": updated,
    "Box score totali": total_box,
    "Nuovi box score": downloaded,
    "Download falliti": len(failed),
    "Date non recuperate": len(failed_dates)
}])

report.to_csv(
    os.path.join(BASE_DIR, "NBA_Pipeline_Status.csv"),
    index=False
)

print("=" * 45)
print("STEP 12 TERMINATO")
print("=" * 45)

print("Partite totali:", total_games)
print("Statistiche individuali:", total_box)
print("Nuovi box score:", downloaded)
print("Download falliti:", failed)
print("Date non recuperate:", failed_dates)

if failed or failed_dates:
    print("ATTENZIONE: aggiornamento parziale.")
else:
    print("Aggiornamento completato.")


In [ ]:
# Advanced features


# ============================================================
# STEP 15 — ADVANCED PLAYER INTELLIGENCE
# NBA DATA SCIENCE ANALYTICS
# ============================================================

import os
import sqlite3
import numpy as np
import pandas as pd
import plotly.express as px

from IPython.display import display

# ============================================================
# 1. CONFIGURAZIONE
# ============================================================

if "BASE_DIR" not in globals():
    raise RuntimeError("Esegui prima STEP 0.")

DB_PATH = os.path.join(
    BASE_DIR, "nba_database.db"
)

CSV_PATH = os.path.join(
    BASE_DIR, "NBA_Advanced_ML_Features.csv"
)

EXCEL_PATH = os.path.join(
    BASE_DIR, "NBA_Advanced_Player_Intelligence.xlsx"
)

# ============================================================
# 2. CARICAMENTO DATABASE
# ============================================================

with sqlite3.connect(DB_PATH) as conn:

    box = pd.read_sql_query(
        "SELECT * FROM espn_player_boxscores",
        conn
    )

    games = pd.read_sql_query(
        "SELECT * FROM espn_games",
        conn
    )

print("Box score originali:", len(box))
print("Partite originali:", len(games))

# ============================================================
# 3. PULIZIA DATI
# ============================================================

for df in [box, games]:

    df["game_id"] = df["game_id"].astype(str)

    df["game_date"] = pd.to_datetime(
        df["game_date"],
        utc=True,
        errors="coerce",
        format="mixed"
    )

for c in ["player_id", "team_id"]:
    box[c] = box[c].astype(str)

for c in ["home_id", "away_id"]:
    games[c] = games[c].astype(str)

numeric_cols = [
    "minutes", "pts", "reb", "ast",
    "stl", "blk", "tov",
    "oreb", "dreb", "pf",
    "fgm", "fga", "fg3m", "fg3a",
    "ftm", "fta", "plus_minus",
    "did_not_play"
]

for c in numeric_cols:

    if c not in box.columns:
        raise ValueError(
            f"Colonna mancante: {c}"
        )

    box[c] = pd.to_numeric(
        box[c],
        errors="coerce"
    )

for c in [
    "home_score",
    "away_score",
    "completed"
]:

    games[c] = pd.to_numeric(
        games[c],
        errors="coerce"
    )

games = games[
    (games["completed"] == 1)
    & games["game_date"].notna()
].dropna(
    subset=["home_score", "away_score"]
).drop_duplicates(
    "game_id"
).copy()

box = box[
    (box["minutes"] > 0)
    & (box["did_not_play"] == 0)
    & box["game_id"].isin(games["game_id"])
    & box["game_date"].notna()
].dropna(
    subset=[
        "pts", "reb", "ast", "fg3m"
    ]
).drop_duplicates(
    ["game_id", "player_id"]
).copy()

box["PRA"] = (
    box["pts"]
    + box["reb"]
    + box["ast"]
)

print("Box score validi:", len(box))

# ============================================================
# 4. FUNZIONE DIVISIONE SICURA
# ============================================================

def safe_divide(a, b):

    return a / b.replace(0, np.nan)

# ============================================================
# 5. ADVANCED PLAYER STATISTICS
# ============================================================

# True Shooting Percentage

box["TS_pct"] = 100 * safe_divide(
    box["pts"],
    2 * (
        box["fga"]
        + 0.44 * box["fta"]
    )
)

# Effective Field Goal Percentage

box["eFG_pct"] = 100 * safe_divide(
    box["fgm"] + 0.5 * box["fg3m"],
    box["fga"]
)

# Statistiche per minuto

box["PTS_per_min"] = safe_divide(
    box["pts"],
    box["minutes"]
)

box["REB_per_min"] = safe_divide(
    box["reb"],
    box["minutes"]
)

box["AST_per_min"] = safe_divide(
    box["ast"],
    box["minutes"]
)

box["PRA_per_min"] = safe_divide(
    box["PRA"],
    box["minutes"]
)

# Coinvolgimento offensivo stimato

box["Offensive_Load"] = (
    box["fga"]
    + 0.44 * box["fta"]
    + box["tov"]
)

# ============================================================
# 6. TEAM STATISTICS
# ============================================================

team_stats = box.groupby(
    ["game_id", "team_id"],
    as_index=False
).agg(
    Team_PTS=("pts", "sum"),
    Team_FGA=("fga", lambda s: s.sum(min_count=1)),
    Team_FTA=("fta", lambda s: s.sum(min_count=1)),
    Team_OREB=("oreb", lambda s: s.sum(min_count=1)),
    Team_TOV=("tov", lambda s: s.sum(min_count=1)),
    Team_MIN=("minutes", "sum")
)

# Stima dei possessi

team_stats["Possessions"] = (
    team_stats["Team_FGA"]
    + 0.44 * team_stats["Team_FTA"]
    - team_stats["Team_OREB"]
    + team_stats["Team_TOV"]
)

team_stats["Possessions"] = (
    team_stats["Possessions"]
    .where(team_stats["Possessions"] > 0)
)

team_stats["Offensive_Rating"] = (
    100 * safe_divide(
        team_stats["Team_PTS"],
        team_stats["Possessions"]
    )
)

team_stats["Team_Offensive_Load"] = (
    team_stats["Team_FGA"]
    + 0.44 * team_stats["Team_FTA"]
    + team_stats["Team_TOV"]
)

# ============================================================
# 7. CONTESTO PARTITA
# ============================================================

context = games[[
    "game_id",
    "home_id",
    "away_id",
    "home_score",
    "away_score"
]].copy()

box = box.merge(
    context,
    on="game_id",
    how="inner",
    validate="many_to_one"
)

box = box[
    (box["team_id"] == box["home_id"])
    | (box["team_id"] == box["away_id"])
].copy()

box["home"] = (
    box["team_id"] == box["home_id"]
).astype(int)

box["opponent_id"] = np.where(
    box["home"] == 1,
    box["away_id"],
    box["home_id"]
)

box["Team_Score"] = np.where(
    box["home"] == 1,
    box["home_score"],
    box["away_score"]
)

box["Opponent_Score"] = np.where(
    box["home"] == 1,
    box["away_score"],
    box["home_score"]
)

box = box.merge(
    team_stats,
    on=["game_id", "team_id"],
    how="left",
    validate="many_to_one"
)

opponent_stats = team_stats[[
    "game_id",
    "team_id",
    "Possessions"
]].rename(columns={
    "team_id": "opponent_id",
    "Possessions": "Opponent_Possessions"
})

box = box.merge(
    opponent_stats,
    on=["game_id", "opponent_id"],
    how="left",
    validate="many_to_one"
)

box["Defensive_Rating"] = (
    100 * safe_divide(
        box["Opponent_Score"],
        box["Opponent_Possessions"]
    )
)

box["Pace"] = safe_divide(
    48 * (
        box["Possessions"]
        + box["Opponent_Possessions"]
    ) / 2,
    box["Team_MIN"] / 5
)

# Usage proxy normalizzato per minuti.

box["Usage_Proxy"] = (
    100 * safe_divide(
        box["Offensive_Load"]
        * (box["Team_MIN"] / 5),
        box["minutes"]
        * box["Team_Offensive_Load"]
    )
)

# ============================================================
# 8. ORDINAMENTO TEMPORALE
# ============================================================

box = box.sort_values(
    ["player_id", "game_date", "game_id"]
).reset_index(drop=True)

group = box.groupby(
    "player_id",
    sort=False
)

# ============================================================
# 9. ROLLING FEATURES SENZA DATA LEAKAGE
# ============================================================

# shift(1) esclude la partita che vogliamo prevedere.

rolling_stats = [
    "pts", "reb", "ast", "fg3m",
    "PRA", "minutes",
    "TS_pct", "eFG_pct",
    "PTS_per_min",
    "REB_per_min",
    "AST_per_min",
    "PRA_per_min",
    "Usage_Proxy",
    "Pace",
    "Offensive_Rating",
    "Defensive_Rating"
]

feature_data = {}

for stat in rolling_stats:

    for window in [5, 10, 20]:

        feature_data[
            f"{stat}_avg{window}"
        ] = group[stat].transform(
            lambda s: (
                s.shift(1)
                .rolling(
                    window,
                    min_periods=window
                )
                .mean()
            )
        )

    feature_data[
        f"{stat}_std10"
    ] = group[stat].transform(
        lambda s: (
            s.shift(1)
            .rolling(
                10,
                min_periods=10
            )
            .std()
        )
    )

    feature_data[
        f"{stat}_last1"
    ] = group[stat].shift(1)

features = pd.DataFrame(
    feature_data,
    index=box.index
)

box = pd.concat(
    [box, features],
    axis=1
)

# ============================================================
# 10. MOMENTUM E CONSISTENCY
# ============================================================

for stat in [
    "pts", "reb", "ast",
    "fg3m", "PRA", "minutes"
]:

    box[f"{stat}_momentum"] = (
        box[f"{stat}_avg5"]
        - box[f"{stat}_avg20"]
    )

    box[f"{stat}_momentum_pct"] = (
        100 * safe_divide(
            box[f"{stat}_momentum"],
            box[f"{stat}_avg20"]
        )
    )

    box[f"{stat}_variation"] = safe_divide(
        box[f"{stat}_std10"],
        box[f"{stat}_avg10"]
    )

    box[f"{stat}_consistency"] = (
        100 / (
            1 + box[f"{stat}_variation"]
        )
    )

# ============================================================
# 11. RIPOSO E BACK-TO-BACK
# ============================================================

box["local_day"] = (
    box["game_date"]
    .dt.tz_convert("America/New_York")
    .dt.tz_localize(None)
    .dt.normalize()
)

previous_day = box.groupby(
    "player_id"
)["local_day"].shift(1)

box["Days_Between_Games"] = (
    box["local_day"] - previous_day
).dt.days

box["Rest_Days"] = (
    box["Days_Between_Games"] - 1
).clip(lower=0, upper=30)

box["Back_to_Back"] = (
    box["Days_Between_Games"] == 1
).astype(int)

box["Games_Previous"] = (
    box.groupby("player_id").cumcount()
)

# ============================================================
# 12. HOME / AWAY PERFORMANCE
# ============================================================

for stat in ["pts", "reb", "ast", "PRA"]:

    box[f"{stat}_venue_avg10"] = (
        box.groupby(
            ["player_id", "home"]
        )[stat].transform(
            lambda s: (
                s.shift(1)
                .rolling(
                    10,
                    min_periods=3
                )
                .mean()
            )
        )
    )

# ============================================================
# 13. DIFESA E PACE AVVERSARIA
# ============================================================

team_context = box[[
    "game_id",
    "game_date",
    "team_id",
    "Team_Score",
    "Opponent_Score",
    "Offensive_Rating",
    "Defensive_Rating",
    "Pace"
]].drop_duplicates(
    ["game_id", "team_id"]
).sort_values(
    ["team_id", "game_date"]
).copy()

team_features = [
    "Team_Score",
    "Opponent_Score",
    "Offensive_Rating",
    "Defensive_Rating",
    "Pace"
]

for stat in team_features:

    team_context[f"pre_{stat}_avg10"] = (
        team_context.groupby(
            "team_id"
        )[stat].transform(
            lambda s: (
                s.shift(1)
                .rolling(
                    10,
                    min_periods=5
                )
                .mean()
            )
        )
    )

opp_columns = [
    f"pre_{s}_avg10"
    for s in team_features
]

opp_context = team_context[
    ["game_id", "team_id"] + opp_columns
].rename(columns={
    "team_id": "opponent_id",
    **{
        c: f"Opp_{c}"
        for c in opp_columns
    }
})

box = box.merge(
    opp_context,
    on=["game_id", "opponent_id"],
    how="left",
    validate="many_to_one"
)

# ============================================================
# 14. SELEZIONE FEATURE MACHINE LEARNING
# ============================================================

feature_cols = list(feature_data.keys())

feature_cols += [
    "home",
    "Rest_Days",
    "Back_to_Back",
    "Games_Previous"
]

for stat in [
    "pts", "reb", "ast",
    "fg3m", "PRA", "minutes"
]:

    feature_cols += [
        f"{stat}_momentum",
        f"{stat}_momentum_pct",
        f"{stat}_variation",
        f"{stat}_consistency"
    ]

for stat in ["pts", "reb", "ast", "PRA"]:

    feature_cols.append(
        f"{stat}_venue_avg10"
    )

feature_cols += [
    f"Opp_{c}"
    for c in opp_columns
]

feature_cols = list(
    dict.fromkeys(feature_cols)
)

target_cols = [
    "pts", "reb", "ast",
    "fg3m", "PRA"
]

metadata = [
    "game_id",
    "game_date",
    "player_id",
    "player_name",
    "team_id",
    "team_name",
    "opponent_id"
]

dataset = box[
    metadata
    + feature_cols
    + target_cols
].copy()

dataset = dataset.replace(
    [np.inf, -np.inf],
    np.nan
)

dataset = dataset[
    dataset["Games_Previous"] >= 20
].dropna(
    subset=target_cols
).sort_values(
    "game_date"
)

if dataset.empty:
    raise ValueError(
        "Dataset avanzato vuoto."
    )

# Le feature mancanti vengono conservate.
# L'imputazione sarà addestrata solo
# sul training set nello STEP 16.

# ============================================================
# 15. PLAYER INTELLIGENCE RANKING
# ============================================================

recent = box[
    box["game_date"]
    >= box["game_date"].max()
    - pd.Timedelta(days=30)
].copy()

ranking = recent.groupby(
    ["player_id", "player_name"],
    as_index=False
).agg(
    Partite=("game_id", "nunique"),
    PTS=("pts", "mean"),
    REB=("reb", "mean"),
    AST=("ast", "mean"),
    PRA=("PRA", "mean"),
    MIN=("minutes", "mean"),
    PTS_per_min=("PTS_per_min", "mean"),
    Usage_Proxy=("Usage_Proxy", "mean"),
    PTS_Momentum=("pts_momentum", "last"),
    PTS_Consistency=("pts_consistency", "last")
)

ranking = ranking[
    ranking["Partite"] >= 5
].sort_values(
    "PRA",
    ascending=False
)

# ============================================================
# 16. CONTROLLO QUALITA'
# ============================================================

quality = pd.DataFrame({
    "Feature": feature_cols,
    "Missing_pct": [
        100 * dataset[c].isna().mean()
        for c in feature_cols
    ]
}).sort_values(
    "Missing_pct",
    ascending=False
)

# ============================================================
# 17. SALVATAGGIO DATASET
# ============================================================

dataset.to_csv(
    CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

import json

FEATURE_PATH = os.path.join(
    BASE_DIR,
    "NBA_Advanced_Feature_List.json"
)

with open(
    FEATURE_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump({
        "features": feature_cols,
        "targets": target_cols,
        "minimum_history": 20
    }, f, indent=2)

with pd.ExcelWriter(
    EXCEL_PATH,
    engine="openpyxl"
) as writer:

    ranking.to_excel(
        writer,
        sheet_name="Player Ranking",
        index=False
    )

    quality.to_excel(
        writer,
        sheet_name="Feature Quality",
        index=False
    )

    sample = dataset.tail(500).copy()

    sample["game_date"] = (
        sample["game_date"].astype(str)
    )

    sample.to_excel(
        writer,
        sheet_name="ML Dataset Sample",
        index=False
    )

# ============================================================
# 18. VISUALIZZAZIONE
# ============================================================

print("\nTOP 20 PLAYER INTELLIGENCE")

display(
    ranking.head(20).round(2)
)

if not ranking.empty:

    fig = px.bar(
        ranking.head(20),
        x="player_name",
        y="PRA",
        color="PTS_Consistency",
        hover_data=[
            "PTS",
            "REB",
            "AST",
            "MIN",
            "Usage_Proxy"
        ],
        title="NBA Advanced Player Ranking",
        template="plotly_dark"
    )

    fig.show()

    fig = px.scatter(
        ranking,
        x="Usage_Proxy",
        y="PTS",
        size="MIN",
        color="PTS_Consistency",
        hover_name="player_name",
        title="Offensive Load vs Scoring",
        template="plotly_dark"
    )

    fig.show()

    momentum = ranking.dropna(
        subset=["PTS_Momentum"]
    ).sort_values(
        "PTS_Momentum",
        ascending=False
    ).head(20)

    if not momentum.empty:

        fig = px.bar(
            momentum,
            x="player_name",
            y="PTS_Momentum",
            title="NBA Player Momentum",
            template="plotly_dark"
        )

        fig.show()

# ============================================================
# 19. RIEPILOGO FINALE
# ============================================================

print("\n" + "=" * 50)
print("STEP 15 COMPLETATO!")
print("=" * 50)

print("Righe dataset ML:", len(dataset))
print("Feature avanzate:", len(feature_cols))
print("Giocatori:", dataset["player_id"].nunique())
print("Partite:", dataset["game_id"].nunique())

print("\nPeriodo storico:")
print(dataset["game_date"].min())
print(dataset["game_date"].max())

print("\nFile salvati:")
print(CSV_PATH)
print(EXCEL_PATH)
print(FEATURE_PATH)

print("\nCONTROLLO FEATURE MANCANTI")

display(
    quality.head(15).round(2)
)

print("\nPRONTO PER STEP 16!")


In [ ]:
# Ensemble model training


# ============================================================
# STEP 16 — ADVANCED MACHINE LEARNING & ENSEMBLE
# NBA DATA SCIENCE ANALYTICS
# MODEL VERSION 3.0
# ============================================================

import os
import json
import time
import joblib
import warnings
import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import Ridge

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from IPython.display import display

# ============================================================
# 1. CONFIGURAZIONE
# ============================================================

if "BASE_DIR" not in globals():
    raise RuntimeError("Esegui prima STEP 0.")

DATA_PATH = os.path.join(
    BASE_DIR,
    "NBA_Advanced_ML_Features.csv"
)

FEATURE_PATH = os.path.join(
    BASE_DIR,
    "NBA_Advanced_Feature_List.json"
)

MODEL_PATH = os.path.join(
    BASE_DIR,
    "nba_player_models_v3.joblib"
)

REPORT_PATH = os.path.join(
    BASE_DIR,
    "NBA_Advanced_Model_Accuracy.xlsx"
)

PREDICTIONS_PATH = os.path.join(
    BASE_DIR,
    "NBA_Advanced_Test_Predictions.csv"
)

RANDOM_STATE = 42

TARGETS = {
    "PTS": "pts",
    "REB": "reb",
    "AST": "ast",
    "3PM": "fg3m",
    "PRA": "PRA"
}

for path in [DATA_PATH, FEATURE_PATH]:

    if not os.path.isfile(path):
        raise FileNotFoundError(path)

START_TIME = time.time()

# ============================================================
# 2. CARICAMENTO DATASET
# ============================================================

df = pd.read_csv(
    DATA_PATH,
    dtype={
        "game_id": str,
        "player_id": str,
        "team_id": str,
        "opponent_id": str
    }
)

with open(
    FEATURE_PATH,
    "r",
    encoding="utf-8"
) as f:

    config = json.load(f)

feature_cols = config["features"]

df["game_date"] = pd.to_datetime(
    df["game_date"],
    utc=True,
    errors="coerce",
    format="mixed"
)

required = (
    feature_cols
    + list(TARGETS.values())
    + ["game_date", "game_id", "player_id"]
)

missing = [
    c for c in required
    if c not in df.columns
]

if missing:
    raise ValueError(
        f"Colonne mancanti: {missing}"
    )

for c in feature_cols + list(TARGETS.values()):

    df[c] = pd.to_numeric(
        df[c],
        errors="coerce"
    )

df = df.replace(
    [np.inf, -np.inf],
    np.nan
)

df = df.dropna(
    subset=[
        "game_date",
        "game_id",
        "player_id",
        *TARGETS.values()
    ]
).drop_duplicates(
    ["game_id", "player_id"]
).sort_values(
    "game_date"
).reset_index(drop=True)

print("Dataset:", len(df))
print("Feature iniziali:", len(feature_cols))
print("Giocatori:", df["player_id"].nunique())

# ============================================================
# 3. TRAIN / VALIDATION / TEST
# SUDDIVISIONE CRONOLOGICA PER GIORNATA NBA
# ============================================================

df["NBA_Day"] = (
    df["game_date"]
    .dt.tz_convert("America/New_York")
    .dt.strftime("%Y-%m-%d")
)

days = sorted(
    df["NBA_Day"].unique()
)

if len(days) < 30:
    raise ValueError(
        "Storico insufficiente per il backtest."
    )

train_end = int(len(days) * 0.70)
val_end = int(len(days) * 0.85)

train_days = days[:train_end]

val_days = days[
    train_end:val_end
]

test_days = days[val_end:]

train = df[
    df["NBA_Day"].isin(train_days)
].copy()

val = df[
    df["NBA_Day"].isin(val_days)
].copy()

test = df[
    df["NBA_Day"].isin(test_days)
].copy()

if min(len(train), len(val), len(test)) == 0:
    raise ValueError("Split temporale non valido.")

# Rimuoviamo feature completamente vuote
# utilizzando esclusivamente il training set.

feature_cols = [
    c for c in feature_cols
    if train[c].notna().any()
]

# Evitiamo che le statistiche target
# siano incluse direttamente tra le feature.

leakage = set(feature_cols).intersection(
    TARGETS.values()
)

if leakage:
    raise ValueError(
        f"Possibile data leakage: {leakage}"
    )

X_train = train[feature_cols]
X_val = val[feature_cols]
X_test = test[feature_cols]

print("\nTEMPORAL SPLIT")

print("Training:", len(train))
print("Validation:", len(val))
print("Test:", len(test))

print("\nTraining:", train_days[0], train_days[-1])
print("Validation:", val_days[0], val_days[-1])
print("Test:", test_days[0], test_days[-1])

print("Feature utilizzate:", len(feature_cols))

# ============================================================
# 4. DEFINIZIONE MODELLI
# ============================================================

def build_models():

    return {

        "Ridge": Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "model",
                Ridge(alpha=10.0)
            )
        ]),

        "Random Forest": Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "model",
                RandomForestRegressor(
                    n_estimators=150,
                    max_depth=12,
                    min_samples_leaf=8,
                    max_features=0.7,
                    random_state=RANDOM_STATE,
                    n_jobs=2
                )
            )
        ]),

        "Extra Trees": Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "model",
                ExtraTreesRegressor(
                    n_estimators=150,
                    max_depth=14,
                    min_samples_leaf=8,
                    max_features=0.8,
                    random_state=RANDOM_STATE,
                    n_jobs=2
                )
            )
        ]),

        "XGBoost": XGBRegressor(
            n_estimators=250,
            learning_rate=0.04,
            max_depth=5,
            subsample=0.85,
            colsample_bytree=0.8,
            reg_lambda=5,
            objective="reg:squarederror",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=2
        ),

        "LightGBM": LGBMRegressor(
            n_estimators=250,
            learning_rate=0.04,
            num_leaves=31,
            max_depth=7,
            min_child_samples=30,
            subsample=0.85,
            subsample_freq=1,
            colsample_bytree=0.8,
            reg_lambda=5,
            random_state=RANDOM_STATE,
            n_jobs=2,
            verbosity=-1
        )
    }

# ============================================================
# 5. METRICHE DI VALUTAZIONE
# ============================================================

def evaluate(y, prediction):

    y = np.asarray(y, dtype=float)

    prediction = np.maximum(
        np.asarray(prediction, dtype=float),
        0
    )

    error = prediction - y

    return {
        "MAE": mean_absolute_error(
            y, prediction
        ),
        "RMSE": np.sqrt(
            mean_squared_error(y, prediction)
        ),
        "Bias": float(error.mean()),
        "Median_Error": float(
            np.median(np.abs(error))
        ),
        "R2": r2_score(y, prediction),
        "Campione": len(y)
    }

# ============================================================
# 6. BASELINE
# ============================================================

def baseline_predict(data, target, fallback):

    col = f"{target}_avg10"

    if col not in data.columns:

        raise ValueError(
            f"Baseline mancante: {col}"
        )

    return np.maximum(
        data[col]
        .fillna(fallback)
        .to_numpy(dtype=float),
        0
    )

# ============================================================
# 7. TRAINING E VALIDATION
# ============================================================

validation_results = []
test_results = []
selected_models = {}
all_predictions = []
importance_rows = []

for target_name, target_col in TARGETS.items():

    print("\n" + "=" * 55)
    print("TRAINING TARGET:", target_name)
    print("=" * 55)

    y_train = train[target_col]
    y_val = val[target_col]
    y_test = test[target_col]

    fallback = float(y_train.mean())

    candidates = build_models()

    fitted_models = {
        "Last 10 Baseline": None
    }

    val_predictions = {}
    val_scores = {}

    # --------------------------------------------------------
    # BASELINE
    # --------------------------------------------------------

    baseline_val = baseline_predict(
        val,
        target_col,
        fallback
    )

    val_predictions[
        "Last 10 Baseline"
    ] = baseline_val

    score = evaluate(
        y_val,
        baseline_val
    )

    val_scores[
        "Last 10 Baseline"
    ] = score["MAE"]

    validation_results.append({
        "Target": target_name,
        "Model": "Last 10 Baseline",
        **score
    })

    print(
        "Baseline MAE:",
        round(score["MAE"], 4)
    )

    # --------------------------------------------------------
    # ADDESTRAMENTO MODELLI
    # --------------------------------------------------------

    for name, model in candidates.items():

        model_start = time.time()

        print(f"\nTraining {name}...")

        model.fit(
            X_train,
            y_train
        )

        prediction = np.maximum(
            model.predict(X_val),
            0
        )

        score = evaluate(
            y_val,
            prediction
        )

        fitted_models[name] = model

        val_predictions[name] = prediction

        val_scores[name] = score["MAE"]

        validation_results.append({
            "Target": target_name,
            "Model": name,
            **score
        })

        print(
            f"{name} MAE:",
            round(score["MAE"], 4)
        )

        print(
            "Tempo:",
            round(time.time() - model_start, 1),
            "secondi"
        )

    # ========================================================
    # 8. WEIGHTED ENSEMBLE
    # ========================================================

    ranked = sorted(
        val_scores,
        key=val_scores.get
    )

    top_models = ranked[:3]

    inverse_errors = np.array([
        1 / max(val_scores[name], 1e-8)
        for name in top_models
    ])

    weights = (
        inverse_errors
        / inverse_errors.sum()
    )

    ensemble_val = np.zeros(
        len(val)
    )

    for name, weight in zip(
        top_models,
        weights
    ):

        ensemble_val += (
            weight
            * val_predictions[name]
        )

    ensemble_score = evaluate(
        y_val,
        ensemble_val
    )

    val_scores[
        "Weighted Ensemble"
    ] = ensemble_score["MAE"]

    validation_results.append({
        "Target": target_name,
        "Model": "Weighted Ensemble",
        **ensemble_score
    })

    print(
        "\nEnsemble MAE:",
        round(
            ensemble_score["MAE"],
            4
        )
    )

    # ========================================================
    # 9. SELEZIONE MODELLO
    # SOLO SULLA VALIDATION
    # ========================================================

    winner = min(
        val_scores,
        key=val_scores.get
    )

    print("\nMODELLO SELEZIONATO:", winner)

    # ========================================================
    # 10. REFIT TRAIN + VALIDATION
    # ========================================================

    development = pd.concat(
        [train, val],
        ignore_index=True
    )

    X_dev = development[feature_cols]
    y_dev = development[target_col]

    final_fallback = float(y_dev.mean())

    if winner == "Weighted Ensemble":

        members = top_models

        final_weights = [
            float(w) for w in weights
        ]

    else:

        members = [winner]
        final_weights = [1.0]

    final_models = {}

    test_prediction = np.zeros(
        len(test)
    )

    for name, weight in zip(
        members,
        final_weights
    ):

        if name == "Last 10 Baseline":

            prediction = baseline_predict(
                test,
                target_col,
                final_fallback
            )

            final_models[name] = None

        else:

            model = build_models()[name]

            model.fit(
                X_dev,
                y_dev
            )

            prediction = np.maximum(
                model.predict(X_test),
                0
            )

            final_models[name] = model

        test_prediction += (
            weight * prediction
        )

    # ========================================================
    # 11. TEST FINALE OUT-OF-SAMPLE
    # ========================================================

    test_score = evaluate(
        y_test,
        test_prediction
    )

    baseline_test = baseline_predict(
        test,
        target_col,
        final_fallback
    )

    baseline_score = evaluate(
        y_test,
        baseline_test
    )

    improvement = 100 * (
        baseline_score["MAE"]
        - test_score["MAE"]
    ) / max(
        baseline_score["MAE"],
        1e-8
    )

    test_results.append({
        "Target": target_name,
        "Selected_Model": winner,
        **test_score,
        "Baseline_MAE": baseline_score["MAE"],
        "Improvement_pct": improvement
    })

    selected_models[target_name] = {
        "name": winner,
        "members": members,
        "weights": final_weights,
        "models": final_models,
        "target_column": target_col,
        "baseline_column": f"{target_col}_avg10",
        "fallback": final_fallback
    }

    # ========================================================
    # 12. SALVATAGGIO PREVISIONI TEST
    # ========================================================

    pred_df = test[[
        "game_id",
        "game_date",
        "player_id",
        "player_name",
        "team_name"
    ]].copy()

    pred_df["Target"] = target_name

    pred_df["Actual"] = (
        y_test.to_numpy()
    )

    pred_df["Forecast"] = (
        test_prediction
    )

    pred_df["Baseline"] = (
        baseline_test
    )

    pred_df["Error"] = (
        pred_df["Forecast"]
        - pred_df["Actual"]
    )

    pred_df["Abs_Error"] = (
        pred_df["Error"].abs()
    )

    pred_df["Model"] = winner

    all_predictions.append(pred_df)

    # ========================================================
    # 13. FEATURE IMPORTANCE
    # ========================================================

    for name, model in final_models.items():

        if model is None:
            continue

        estimator = model

        if isinstance(model, Pipeline):

            estimator = model.named_steps[
                "model"
            ]

        if hasattr(
            estimator,
            "feature_importances_"
        ):

            importance = (
                estimator.feature_importances_
            )

            total = np.sum(importance)

            if total > 0:
                importance = importance / total

            if len(importance) != len(feature_cols):
                continue

            for feature, value in zip(
                feature_cols,
                importance
            ):

                importance_rows.append({
                    "Target": target_name,
                    "Model": name,
                    "Feature": feature,
                    "Importance": float(value)
                })

    print("\nTEST MAE:", round(test_score["MAE"], 4))

    print(
        "Miglioramento vs baseline:",
        round(improvement, 2),
        "%"
    )

# ============================================================
# 14. CREAZIONE REPORT
# ============================================================

validation_df = pd.DataFrame(
    validation_results
).sort_values(
    ["Target", "MAE"]
)

test_df = pd.DataFrame(
    test_results
)

predictions = pd.concat(
    all_predictions,
    ignore_index=True
)

importance_df = pd.DataFrame(
    importance_rows,
    columns=[
        "Target",
        "Model",
        "Feature",
        "Importance"
    ]
)

if not importance_df.empty:

    importance_df = importance_df.sort_values(
        ["Target", "Model", "Importance"],
        ascending=[True, True, False]
    )

# ============================================================
# 15. PERFORMANCE PER GIOCATORE
# ============================================================

player_accuracy = predictions.groupby(
    [
        "player_id",
        "player_name",
        "Target"
    ],
    as_index=False
).agg(
    Partite=("game_id", "nunique"),
    MAE=("Abs_Error", "mean"),
    Bias=("Error", "mean"),
    Actual_Mean=("Actual", "mean"),
    Forecast_Mean=("Forecast", "mean")
)

player_accuracy = player_accuracy[
    player_accuracy["Partite"] >= 5
].sort_values(
    ["Target", "MAE"]
)

# ============================================================
# 16. ENSEMBLE WEIGHTS
# ============================================================

weight_rows = []

for target, info in selected_models.items():

    for name, weight in zip(
        info["members"],
        info["weights"]
    ):

        weight_rows.append({
            "Target": target,
            "Selected_Model": info["name"],
            "Member": name,
            "Weight": weight
        })

weights_df = pd.DataFrame(
    weight_rows
)

# ============================================================
# 17. SALVATAGGIO MODELLO V3
# ============================================================

package_v3 = {
    "model_version": "3.0",
    "models": selected_models,
    "features": feature_cols,
    "targets": TARGETS,
    "minimum_history": config.get(
        "minimum_history", 20
    ),
    "training_cutoff": (
        development["game_date"]
        .max()
        .isoformat()
    ),
    "train_period": [
        train_days[0],
        train_days[-1]
    ],
    "validation_period": [
        val_days[0],
        val_days[-1]
    ],
    "test_period": [
        test_days[0],
        test_days[-1]
    ],
    "validation": (
        validation_df.to_dict("records")
    ),
    "test": test_df.to_dict("records"),
    "created_at": (
        pd.Timestamp.now(tz="UTC")
        .isoformat()
    )
}

joblib.dump(
    package_v3,
    MODEL_PATH + ".tmp",
    compress=3
)

os.replace(
    MODEL_PATH + ".tmp",
    MODEL_PATH
)

# ============================================================
# 18. ESPORTAZIONE CSV ED EXCEL
# ============================================================

predictions["game_date"] = (
    predictions["game_date"].astype(str)
)

predictions.to_csv(
    PREDICTIONS_PATH,
    index=False,
    encoding="utf-8-sig"
)

split_report = pd.DataFrame([
    {
        "Dataset": "Training",
        "Rows": len(train),
        "Start": train_days[0],
        "End": train_days[-1]
    },
    {
        "Dataset": "Validation",
        "Rows": len(val),
        "Start": val_days[0],
        "End": val_days[-1]
    },
    {
        "Dataset": "Test",
        "Rows": len(test),
        "Start": test_days[0],
        "End": test_days[-1]
    }
])

with pd.ExcelWriter(
    REPORT_PATH,
    engine="openpyxl"
) as writer:

    sheets = {
        "Validation": validation_df,
        "Test": test_df,
        "Predictions": predictions,
        "Player Accuracy": player_accuracy,
        "Feature Importance": importance_df,
        "Ensemble Weights": weights_df,
        "Temporal Split": split_report
    }

    for name, data in sheets.items():

        data.to_excel(
            writer,
            sheet_name=name,
            index=False
        )

# ============================================================
# 19. VISUALIZZAZIONE
# ============================================================

print("\nVALIDATION RESULTS")

display(
    validation_df.round(4)
)

print("\nFINAL TEST RESULTS")

display(
    test_df.round(4)
)

fig = px.bar(
    validation_df,
    x="Model",
    y="MAE",
    color="Model",
    facet_col="Target",
    facet_col_wrap=2,
    title="NBA V3 — Model Validation",
    template="plotly_dark"
)

fig.update_yaxes(matches=None)

fig.update_layout(
    height=1000,
    showlegend=False
)

fig.show()

# Baseline vs modello selezionato

fig = px.bar(
    test_df,
    x="Target",
    y=["Baseline_MAE", "MAE"],
    barmode="group",
    title="Baseline vs Advanced ML",
    template="plotly_dark"
)

fig.show()

# Miglioramento percentuale

fig = px.bar(
    test_df,
    x="Target",
    y="Improvement_pct",
    color="Improvement_pct",
    title="Model Improvement vs Baseline (%)",
    template="plotly_dark"
)

fig.add_hline(y=0)

fig.show()

# Feature importance per punti

if not importance_df.empty:

    pts_importance = importance_df[
        importance_df["Target"] == "PTS"
    ].copy()

    if not pts_importance.empty:

        pts_importance = (
            pts_importance
            .sort_values(
                "Importance",
                ascending=False
            )
            .groupby("Model")
            .head(15)
        )

        fig = px.bar(
            pts_importance,
            x="Importance",
            y="Feature",
            color="Model",
            orientation="h",
            barmode="group",
            title="PTS — Feature Importance",
            template="plotly_dark"
        )

        fig.update_layout(height=700)

        fig.show()

# ============================================================
# 20. RIEPILOGO FINALE
# ============================================================

elapsed = (
    time.time() - START_TIME
) / 60

print("\n" + "=" * 55)
print("STEP 16 COMPLETATO!")
print("=" * 55)

print("Versione modello: 3.0")
print("Feature:", len(feature_cols))
print("Training rows:", len(train))
print("Validation rows:", len(val))
print("Test rows:", len(test))

print(
    "Tempo totale:",
    round(elapsed, 2),
    "minuti"
)

print("\nMODELLI SELEZIONATI")

for target, info in selected_models.items():

    print(
        target,
        "->",
        info["name"]
    )

print("\nFILE SALVATI")

print(MODEL_PATH)
print(REPORT_PATH)
print(PREDICTIONS_PATH)

print("\nPRONTO PER STEP 17!")


In [ ]:
# Player forecast engine


# ============================================================
# STEP 17 — ADVANCED FORECAST ENGINE V3
# NBA DATA SCIENCE ANALYTICS
# ============================================================

import os
import sqlite3
import joblib
import numpy as np
import pandas as pd
import plotly.express as px

from google.colab import drive
from IPython.display import display

# ============================================================
# 1. RIPRISTINO AUTOMATICO AMBIENTE
# ============================================================

# Preserve the directory selected in the setup cell.
BASE_DIR = globals().get("BASE_DIR") or os.environ.get("NBA_DATA_DIR")
if not BASE_DIR or not os.path.isfile(os.path.join(BASE_DIR, "nba_database.db")):
    raise FileNotFoundError("Run the setup cell and collect the NBA database first.")


DB_PATH = os.path.join(
    BASE_DIR, "nba_database.db"
)

MODEL_PATH = os.path.join(
    BASE_DIR, "nba_player_models_v3.joblib"
)

CSV_PATH = os.path.join(
    BASE_DIR, "NBA_Forecast_Engine_V3.csv"
)

EXCEL_PATH = os.path.join(
    BASE_DIR, "NBA_Forecast_Engine_V3.xlsx"
)

ARCHIVE_DIR = os.path.join(
    BASE_DIR, "Forecast_Archive_V3"
)

os.makedirs(ARCHIVE_DIR, exist_ok=True)

if not os.path.isfile(MODEL_PATH):
    raise FileNotFoundError(
        "Modello V3 mancante. Completa STEP 16."
    )

NOW = pd.Timestamp.now(tz="UTC")

# ============================================================
# 2. CARICAMENTO MODELLO V3
# ============================================================

package_v3 = joblib.load(MODEL_PATH)

feature_cols = package_v3["features"]
models = package_v3["models"]

MIN_HISTORY = package_v3.get(
    "minimum_history", 20
)

print("Modello:", package_v3["model_version"])
print("Feature richieste:", len(feature_cols))

# ============================================================
# 3. CARICAMENTO DATABASE
# ============================================================

with sqlite3.connect(DB_PATH) as conn:

    box = pd.read_sql_query(
        "SELECT * FROM espn_player_boxscores",
        conn
    )

    games = pd.read_sql_query(
        "SELECT * FROM espn_games",
        conn
    )

for df in [box, games]:

    df["game_id"] = df["game_id"].astype(str)

    df["game_date"] = pd.to_datetime(
        df["game_date"],
        utc=True,
        errors="coerce",
        format="mixed"
    )

for c in ["player_id", "team_id"]:
    box[c] = box[c].astype(str)

for c in ["home_id", "away_id"]:
    games[c] = games[c].astype(str)

numeric = [
    "minutes", "pts", "reb", "ast",
    "fg3m", "fgm", "fga", "fta",
    "oreb", "tov", "did_not_play"
]

for c in numeric:

    box[c] = pd.to_numeric(
        box[c], errors="coerce"
    )

for c in [
    "home_score", "away_score", "completed"
]:

    games[c] = pd.to_numeric(
        games[c], errors="coerce"
    )

games = games.drop_duplicates("game_id")

completed = games[
    (games["completed"] == 1)
    & (games["game_date"] < NOW)
].dropna(
    subset=["home_score", "away_score"]
).copy()

box = box[
    (box["minutes"] > 0)
    & (box["did_not_play"] == 0)
    & box["game_id"].isin(completed["game_id"])
    & (box["game_date"] < NOW)
].dropna(
    subset=["pts", "reb", "ast", "fg3m"]
).drop_duplicates(
    ["game_id", "player_id"]
).copy()

# ============================================================
# 4. STATISTICHE AVANZATE
# ============================================================

def safe_divide(a, b):

    return a / b.replace(0, np.nan)

box["PRA"] = (
    box["pts"] + box["reb"] + box["ast"]
)

box["TS_pct"] = 100 * safe_divide(
    box["pts"],
    2 * (box["fga"] + 0.44 * box["fta"])
)

box["eFG_pct"] = 100 * safe_divide(
    box["fgm"] + 0.5 * box["fg3m"],
    box["fga"]
)

for stat, name in [
    ("pts", "PTS"),
    ("reb", "REB"),
    ("ast", "AST"),
    ("PRA", "PRA")
]:

    box[f"{name}_per_min"] = safe_divide(
        box[stat], box["minutes"]
    )

box["Offensive_Load"] = (
    box["fga"]
    + 0.44 * box["fta"]
    + box["tov"]
)

# ============================================================
# 5. TEAM ANALYTICS
# ============================================================

team_stats = box.groupby(
    ["game_id", "team_id"],
    as_index=False
).agg(
    Team_PTS=("pts", "sum"),
    Team_FGA=("fga", lambda s: s.sum(min_count=1)),
    Team_FTA=("fta", lambda s: s.sum(min_count=1)),
    Team_OREB=("oreb", lambda s: s.sum(min_count=1)),
    Team_TOV=("tov", lambda s: s.sum(min_count=1)),
    Team_MIN=("minutes", "sum")
)

team_stats["Possessions"] = (
    team_stats["Team_FGA"]
    + 0.44 * team_stats["Team_FTA"]
    - team_stats["Team_OREB"]
    + team_stats["Team_TOV"]
)

team_stats["Possessions"] = (
    team_stats["Possessions"]
    .where(team_stats["Possessions"] > 0)
)

team_stats["Offensive_Rating"] = (
    100 * safe_divide(
        team_stats["Team_PTS"],
        team_stats["Possessions"]
    )
)

team_stats["Team_Offensive_Load"] = (
    team_stats["Team_FGA"]
    + 0.44 * team_stats["Team_FTA"]
    + team_stats["Team_TOV"]
)

context = completed[[
    "game_id", "home_id", "away_id",
    "home_score", "away_score"
]]

box = box.merge(
    context,
    on="game_id",
    how="inner"
)

box = box[
    (box["team_id"] == box["home_id"])
    | (box["team_id"] == box["away_id"])
].copy()

box["home"] = (
    box["team_id"] == box["home_id"]
).astype(int)

box["opponent_id"] = np.where(
    box["home"] == 1,
    box["away_id"],
    box["home_id"]
)

box["Team_Score"] = np.where(
    box["home"] == 1,
    box["home_score"],
    box["away_score"]
)

box["Opponent_Score"] = np.where(
    box["home"] == 1,
    box["away_score"],
    box["home_score"]
)

box = box.merge(
    team_stats,
    on=["game_id", "team_id"],
    how="left"
)

opp_stats = team_stats[[
    "game_id", "team_id", "Possessions"
]].rename(columns={
    "team_id": "opponent_id",
    "Possessions": "Opponent_Possessions"
})

box = box.merge(
    opp_stats,
    on=["game_id", "opponent_id"],
    how="left"
)

box["Defensive_Rating"] = (
    100 * safe_divide(
        box["Opponent_Score"],
        box["Opponent_Possessions"]
    )
)

box["Pace"] = safe_divide(
    48 * (
        box["Possessions"]
        + box["Opponent_Possessions"]
    ) / 2,
    box["Team_MIN"] / 5
)

box["Usage_Proxy"] = (
    100 * safe_divide(
        box["Offensive_Load"]
        * (box["Team_MIN"] / 5),
        box["minutes"]
        * box["Team_Offensive_Load"]
    )
)

box = box.sort_values(
    ["game_date", "game_id"]
)

# ============================================================
# 6. STORICO GIOCATORI E SQUADRE
# ============================================================

histories = {
    pid: group.copy()
    for pid, group in box.groupby("player_id")
}

latest_team = box.drop_duplicates(
    "player_id",
    keep="last"
)

latest_team = latest_team[
    latest_team["game_date"]
    >= NOW - pd.Timedelta(days=90)
]

team_context = box[[
    "game_id",
    "game_date",
    "team_id",
    "Team_Score",
    "Opponent_Score",
    "Offensive_Rating",
    "Defensive_Rating",
    "Pace"
]].drop_duplicates(
    ["game_id", "team_id"]
).sort_values("game_date")

team_histories = {
    tid: group.copy()
    for tid, group in team_context.groupby("team_id")
}

# ============================================================
# 7. PARTITE FUTURE
# ============================================================

future = games[
    (games["completed"] == 0)
    & (games["status"] == "STATUS_SCHEDULED")
    & (games["game_date"] > NOW)
    & (
        games["game_date"]
        <= NOW + pd.Timedelta(days=3)
    )
].sort_values("game_date")

if future.empty:

    raise RuntimeError(
        "Nessuna partita futura. Esegui STEP 12."
    )

print("Partite future:", len(future))

# ============================================================
# 8. FEATURE ENGINEERING V3
# ============================================================

rolling_stats = [
    "pts", "reb", "ast", "fg3m",
    "PRA", "minutes",
    "TS_pct", "eFG_pct",
    "PTS_per_min",
    "REB_per_min",
    "AST_per_min",
    "PRA_per_min",
    "Usage_Proxy",
    "Pace",
    "Offensive_Rating",
    "Defensive_Rating"
]

momentum_stats = [
    "pts", "reb", "ast",
    "fg3m", "PRA", "minutes"
]

team_features = [
    "Team_Score",
    "Opponent_Score",
    "Offensive_Rating",
    "Defensive_Rating",
    "Pace"
]

def scalar_divide(a, b):

    if pd.isna(b) or b == 0:
        return np.nan

    return a / b


def build_advanced_features(
    history, opponent_id, date, home
):

    history = history[
        history["game_date"] < NOW
    ].sort_values("game_date")

    if len(history) < MIN_HISTORY:
        return None

    row = {}

    for stat in rolling_stats:

        for window in [5, 10, 20]:

            values = history[stat].tail(window)

            row[f"{stat}_avg{window}"] = (
                values.mean()
                if values.count() >= window
                else np.nan
            )

        values = history[stat].tail(10)

        row[f"{stat}_std10"] = (
            values.std(ddof=1)
            if values.count() >= 10
            else np.nan
        )

        row[f"{stat}_last1"] = (
            history[stat].iloc[-1]
        )

    # Momentum e consistency

    for stat in momentum_stats:

        momentum = (
            row[f"{stat}_avg5"]
            - row[f"{stat}_avg20"]
        )

        variation = scalar_divide(
            row[f"{stat}_std10"],
            row[f"{stat}_avg10"]
        )

        row[f"{stat}_momentum"] = momentum

        row[f"{stat}_momentum_pct"] = (
            100 * scalar_divide(
                momentum,
                row[f"{stat}_avg20"]
            )
        )

        row[f"{stat}_variation"] = variation

        row[f"{stat}_consistency"] = (
            100 / (1 + variation)
        )

    # Riposo e fattore campo

    game_day = date.tz_convert(
        "America/New_York"
    ).date()

    last_day = history.iloc[-1][
        "game_date"
    ].tz_convert(
        "America/New_York"
    ).date()

    days = (game_day - last_day).days

    row["home"] = int(home)

    row["Rest_Days"] = np.clip(
        days - 1, 0, 30
    )

    row["Back_to_Back"] = int(days == 1)

    row["Games_Previous"] = len(history)

    # Performance casa / trasferta

    venue = history[
        history["home"] == home
    ].tail(10)

    for stat in ["pts", "reb", "ast", "PRA"]:

        row[f"{stat}_venue_avg10"] = (
            venue[stat].mean()
            if venue[stat].count() >= 3
            else np.nan
        )

    # Contesto avversario

    opponent = team_histories.get(opponent_id)

    for stat in team_features:

        key = f"Opp_pre_{stat}_avg10"

        if opponent is None:

            row[key] = np.nan

        else:

            values = opponent[stat].tail(10)

            row[key] = (
                values.mean()
                if values.count() >= 5
                else np.nan
            )

    return row

# ============================================================
# 9. GENERAZIONE DATASET FUTURO
# ============================================================

rows = []

for game in future.itertuples(index=False):

    matchups = [
        (
            game.home_id,
            game.home_team,
            game.away_id,
            game.away_team,
            1
        ),
        (
            game.away_id,
            game.away_team,
            game.home_id,
            game.home_team,
            0
        )
    ]

    for tid, team, oid, opponent, home in matchups:

        roster = latest_team[
            latest_team["team_id"] == tid
        ]

        for player in roster.itertuples(index=False):

            features = build_advanced_features(
                histories[player.player_id],
                oid,
                game.game_date,
                home
            )

            if features is None:
                continue

            rows.append({
                "game_id": game.game_id,
                "player_id": player.player_id,
                "date": game.game_date,
                "Giocatore": player.player_name,
                "Squadra": team,
                "Avversario": opponent,
                "Sede": "Casa" if home else "Trasferta",
                **features
            })

forecast_v3 = pd.DataFrame(rows)

if forecast_v3.empty:
    raise ValueError("Nessun giocatore idoneo.")

missing = [
    c for c in feature_cols
    if c not in forecast_v3.columns
]

if missing:

    raise ValueError(
        f"Feature incompatibili: {missing}"
    )

X = forecast_v3[feature_cols].replace(
    [np.inf, -np.inf],
    np.nan
)

print("Player-game forecast:", len(X))
print("Feature disponibili:", len(X.columns))

# ============================================================
# 10. MACHINE LEARNING ED ENSEMBLE
# ============================================================

for target, info in models.items():

    prediction = np.zeros(
        len(forecast_v3)
    )

    for name, weight in zip(
        info["members"],
        info["weights"]
    ):

        if name == "Last 10 Baseline":

            member_prediction = (
                forecast_v3[info["baseline_column"]]
                .fillna(info["fallback"])
                .to_numpy(dtype=float)
            )

        else:

            model = info["models"][name]

            member_prediction = model.predict(X)

        prediction += (
            weight * np.maximum(
                member_prediction, 0
            )
        )

    forecast_v3[f"Forecast_{target}"] = prediction

    forecast_v3[f"Model_{target}"] = info["name"]

# Manteniamo il PRA previsto dal modello V3.
# Aggiungiamo anche la somma delle previsioni individuali.

forecast_v3["PRA_Sum"] = (
    forecast_v3["Forecast_PTS"]
    + forecast_v3["Forecast_REB"]
    + forecast_v3["Forecast_AST"]
)

# ============================================================
# 11. CONTROLLO QUALITA'
# ============================================================

TARGETS = ["PTS", "REB", "AST", "3PM", "PRA"]

generated_at = pd.Timestamp.now(tz="UTC")

forecast_v3["generated_at"] = generated_at

forecast_v3 = forecast_v3[
    forecast_v3["date"] > generated_at
].copy()

forecast_v3 = forecast_v3.replace(
    [np.inf, -np.inf],
    np.nan
).dropna(
    subset=[
        f"Forecast_{t}"
        for t in TARGETS
    ]
).drop_duplicates(
    ["game_id", "player_id"]
)

if forecast_v3.empty:
    raise ValueError("Nessun forecast valido.")

forecast_v3["Data Italia"] = (
    forecast_v3["date"]
    .dt.tz_convert("Europe/Rome")
    .dt.strftime("%d/%m/%Y %H:%M")
)

forecast_v3["Model_Version"] = "3.0"

forecast_v3["Roster_Status"] = (
    "Stimato - disponibilità non verificata"
)

# ============================================================
# 12. ARCHIVIAZIONE SQLITE SEPARATA V3
# ============================================================

with sqlite3.connect(DB_PATH) as conn:

    conn.execute("""
        CREATE TABLE IF NOT EXISTS nba_forecast_history_v3 (
            game_id TEXT NOT NULL,
            player_id TEXT NOT NULL,
            generated_at TEXT NOT NULL,
            game_date TEXT NOT NULL,
            player_name TEXT,
            team_name TEXT,
            opponent TEXT,
            forecast_pts REAL,
            forecast_reb REAL,
            forecast_ast REAL,
            forecast_3pm REAL,
            forecast_pra REAL,
            model_version TEXT,
            PRIMARY KEY (
                game_id, player_id, generated_at
            )
        )
    """)

    records = []

    for _, r in forecast_v3.iterrows():

        records.append((
            str(r["game_id"]),
            str(r["player_id"]),
            r["generated_at"].isoformat(),
            r["date"].isoformat(),
            r["Giocatore"],
            r["Squadra"],
            r["Avversario"],
            float(r["Forecast_PTS"]),
            float(r["Forecast_REB"]),
            float(r["Forecast_AST"]),
            float(r["Forecast_3PM"]),
            float(r["Forecast_PRA"]),
            "3.0"
        ))

    conn.executemany("""
        INSERT OR IGNORE INTO nba_forecast_history_v3
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, records)

    total_history = conn.execute("""
        SELECT COUNT(*)
        FROM nba_forecast_history_v3
    """).fetchone()[0]

# ============================================================
# 13. ESPORTAZIONE FORECAST
# ============================================================

columns = [
    "game_id",
    "player_id",
    "date",
    "generated_at",
    "Data Italia",
    "Giocatore",
    "Squadra",
    "Avversario",
    "Sede",
    "Forecast_PTS",
    "Forecast_REB",
    "Forecast_AST",
    "Forecast_3PM",
    "Forecast_PRA",
    "PRA_Sum",
    "minutes_avg10",
    "pts_avg10",
    "pts_momentum",
    "pts_consistency",
    "Usage_Proxy_avg10",
    "TS_pct_avg10",
    "Rest_Days",
    "Back_to_Back",
    "Model_Version",
    "Roster_Status"
]

columns += [
    f"Model_{t}"
    for t in TARGETS
]

export_v3 = forecast_v3[columns].copy()

export_v3 = export_v3.sort_values(
    ["date", "Forecast_PTS"],
    ascending=[True, False]
)

for c in ["date", "generated_at"]:

    export_v3[c] = export_v3[c].map(
        lambda x: x.isoformat()
    )

stamp = generated_at.strftime(
    "%Y%m%d_%H%M%S_%f"
)

archive_path = os.path.join(
    ARCHIVE_DIR,
    f"NBA_V3_{stamp}.csv"
)

export_v3.to_csv(
    archive_path,
    index=False,
    encoding="utf-8-sig"
)

export_v3.to_csv(
    CSV_PATH + ".tmp",
    index=False,
    encoding="utf-8-sig"
)

with pd.ExcelWriter(
    EXCEL_PATH + ".tmp.xlsx",
    engine="openpyxl"
) as writer:

    export_v3.to_excel(
        writer,
        sheet_name="Player Forecast",
        index=False
    )

    pd.DataFrame(
        package_v3["test"]
    ).to_excel(
        writer,
        sheet_name="Model Accuracy",
        index=False
    )

os.replace(CSV_PATH + ".tmp", CSV_PATH)

os.replace(
    EXCEL_PATH + ".tmp.xlsx",
    EXCEL_PATH
)

# ============================================================
# 14. CONFRONTO FORECAST V2 VS V3
# ============================================================

V2_PATH = os.path.join(
    BASE_DIR, "NBA_Forecast_Engine_V2.csv"
)

if os.path.isfile(V2_PATH):

    v2 = pd.read_csv(
        V2_PATH,
        dtype={
            "game_id": str,
            "player_id": str
        }
    )

    keys = ["game_id", "player_id"]

    if all(c in v2.columns for c in keys):

        common = v2.merge(
            export_v3,
            on=keys,
            suffixes=("_V2", "_V3")
        )

        for target in TARGETS:

            common[f"Delta_{target}"] = (
                common[f"Forecast_{target}_V3"]
                - common[f"Forecast_{target}_V2"]
            )

        comparison_path = os.path.join(
            BASE_DIR,
            "NBA_V2_vs_V3_Forecasts.csv"
        )

        common.to_csv(
            comparison_path,
            index=False
        )

        print(
            "Forecast confrontabili V2/V3:",
            len(common)
        )

# ============================================================
# 15. VISUALIZZAZIONE
# ============================================================

print("\nTOP 20 — POINTS FORECAST")

top_pts = export_v3.sort_values(
    "Forecast_PTS",
    ascending=False
).head(20)

display(
    top_pts[[
        "Giocatore",
        "Squadra",
        "Avversario",
        "Forecast_PTS",
        "Forecast_REB",
        "Forecast_AST",
        "Forecast_3PM",
        "Forecast_PRA"
    ]].round(2)
)

fig = px.bar(
    top_pts,
    x="Giocatore",
    y="Forecast_PTS",
    color="Squadra",
    hover_data=[
        "Avversario",
        "Data Italia",
        "Forecast_REB",
        "Forecast_AST",
        "Forecast_PRA"
    ],
    title="NBA V3 — Top Points Forecast",
    template="plotly_dark"
)

fig.update_layout(
    height=550,
    xaxis_tickangle=-45
)

fig.show()

top_pra = export_v3.sort_values(
    "Forecast_PRA",
    ascending=False
).head(20)

fig = px.bar(
    top_pra,
    x="Giocatore",
    y="Forecast_PRA",
    color="Squadra",
    title="NBA V3 — Top PRA Forecast",
    template="plotly_dark"
)

fig.show()

# ============================================================
# 16. RIEPILOGO FINALE
# ============================================================

print("\n" + "=" * 55)
print("STEP 17 COMPLETATO!")
print("=" * 55)

print("Versione: V3.0")
print("Partite future:", export_v3["game_id"].nunique())
print("Giocatori:", export_v3["player_id"].nunique())
print("Forecast generati:", len(export_v3))
print("Forecast storici V3:", total_history)
print("Feature ML:", len(feature_cols))

print("\nMODELLI UTILIZZATI")

for target, info in models.items():

    print(target, "->", info["name"])

print("\nFILE SALVATI")

print(CSV_PATH)
print(EXCEL_PATH)
print(archive_path)

print("\nPRONTO PER STEP 18!")


In [ ]:
# Threshold probability engine


# ============================================================
# STEP 18 — PROBABILITY DISTRIBUTION ENGINE
# NBA DATA SCIENCE ANALYTICS
# VERSION 3.1 — FIXED
# ============================================================

import os
import sqlite3
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets

from google.colab import drive
from IPython.display import display, clear_output

# ============================================================
# 1. RIPRISTINO AUTOMATICO AMBIENTE
# ============================================================

# Preserve the directory selected in the setup cell.
BASE_DIR = globals().get("BASE_DIR") or os.environ.get("NBA_DATA_DIR")
if not BASE_DIR or not os.path.isfile(os.path.join(BASE_DIR, "nba_database.db")):
    raise FileNotFoundError("Run the setup cell and collect the NBA database first.")


print("Google Drive collegato.")
print("Cartella progetto:", BASE_DIR)

# ============================================================
# 2. CONFIGURAZIONE
# ============================================================

HISTORY_PATH = os.path.join(
    BASE_DIR,
    "NBA_Advanced_Test_Predictions.csv"
)

FORECAST_PATH = os.path.join(
    BASE_DIR,
    "NBA_Forecast_Engine_V3.csv"
)

OUTPUT_PATH = os.path.join(
    BASE_DIR,
    "NBA_Probability_Engine_V3.xlsx"
)

CSV_PATH = os.path.join(
    BASE_DIR,
    "NBA_Probability_Engine_V3.csv"
)

SUMMARY_PATH = os.path.join(
    BASE_DIR,
    "NBA_Probability_Summary_V3.csv"
)

DB_PATH = os.path.join(
    BASE_DIR,
    "nba_database.db"
)

ARCHIVE_DIR = os.path.join(
    BASE_DIR,
    "Probability_Archive_V3"
)

os.makedirs(
    ARCHIVE_DIR,
    exist_ok=True
)

TARGETS = [
    "PTS", "REB", "AST", "3PM", "PRA"
]

THRESHOLDS = {
    "PTS": [5, 10, 15, 20, 25, 30, 35, 40, 45, 50],
    "REB": [2, 4, 6, 8, 10, 12, 15, 20],
    "AST": [2, 4, 6, 8, 10, 12, 15],
    "3PM": [1, 2, 3, 4, 5, 6, 7],
    "PRA": [10, 15, 20, 25, 30, 35, 40, 45, 50, 60]
}

INTERVALS = [50, 80, 90, 95]

MIN_PLAYER_SAMPLE = 10
MIN_GROUP_SAMPLE = 50

TOLERANCE = 1e-10

NOW = pd.Timestamp.now(tz="UTC")

for path in [HISTORY_PATH, FORECAST_PATH]:

    if not os.path.isfile(path):

        raise FileNotFoundError(
            f"File mancante: {path}"
        )

# ============================================================
# 3. CARICAMENTO DATI
# ============================================================

history = pd.read_csv(
    HISTORY_PATH,
    dtype={
        "game_id": str,
        "player_id": str
    }
)

forecast = pd.read_csv(
    FORECAST_PATH,
    dtype={
        "game_id": str,
        "player_id": str
    }
)

history["game_date"] = pd.to_datetime(
    history["game_date"],
    utc=True,
    errors="coerce",
    format="mixed"
)

forecast["date"] = pd.to_datetime(
    forecast["date"],
    utc=True,
    errors="coerce",
    format="mixed"
)

for col in ["Actual", "Forecast"]:

    history[col] = pd.to_numeric(
        history[col],
        errors="coerce"
    )

for target in TARGETS:

    col = f"Forecast_{target}"

    if col not in forecast.columns:
        raise ValueError(f"Colonna mancante: {col}")

    forecast[col] = pd.to_numeric(
        forecast[col],
        errors="coerce"
    )

history = history.replace(
    [np.inf, -np.inf],
    np.nan
).dropna(
    subset=[
        "game_date",
        "game_id",
        "Actual",
        "Forecast",
        "player_id",
        "Target"
    ]
)

history = history[
    (history["game_date"] < NOW)
    & history["Target"].isin(TARGETS)
    & (history["Actual"] >= 0)
    & (history["Forecast"] >= 0)
].drop_duplicates(
    ["game_id", "player_id", "Target"]
).sort_values(
    "game_date"
)

forecast = forecast[
    forecast["date"] > NOW
].drop_duplicates(
    ["game_id", "player_id"]
).copy()

if history.empty:
    raise ValueError("Storico errori vuoto.")

if forecast.empty:

    raise ValueError(
        "Forecast scaduti. Riesegui STEP 17."
    )

history["Residual"] = (
    history["Actual"]
    - history["Forecast"]
)

print("\nDATI CARICATI")

print("Errori storici:", len(history))
print("Forecast futuri:", len(forecast))

# ============================================================
# 4. SUDDIVISIONE TEMPORALE
# ============================================================

history["NBA_Day"] = (
    history["game_date"]
    .dt.tz_convert("America/New_York")
    .dt.strftime("%Y-%m-%d")
)

days = sorted(
    history["NBA_Day"].unique()
)

if len(days) < 10:

    raise ValueError(
        "Storico insufficiente per calibrazione."
    )

split = int(len(days) * 0.60)

calibration = history[
    history["NBA_Day"].isin(days[:split])
].copy()

evaluation = history[
    history["NBA_Day"].isin(days[split:])
].copy()

print("\nTEMPORAL SPLIT")

print("CALIBRATION:", len(calibration))
print("EVALUATION:", len(evaluation))

# ============================================================
# 5. GRUPPI STATISTICI
# ============================================================

BINS = {
    "PTS": [0, 5, 10, 15, 20, 25, 30, 40, np.inf],
    "REB": [0, 2, 4, 6, 8, 10, 15, np.inf],
    "AST": [0, 2, 4, 6, 8, 10, 15, np.inf],
    "3PM": [0, 1, 2, 3, 4, 6, np.inf],
    "PRA": [0, 10, 20, 30, 40, 50, 60, np.inf]
}

def get_bucket(target, prediction):

    edges = BINS[target]

    return int(
        np.clip(
            np.searchsorted(
                edges,
                prediction,
                side="right"
            ) - 1,
            0,
            len(edges) - 2
        )
    )

# ============================================================
# 6. COSTRUZIONE DISTRIBUZIONI EMPIRICHE
# ============================================================

def build_error_pools(data):

    pools = {
        "global": {},
        "group": {},
        "player": {}
    }

    temp = data[
        data["Target"].isin(TARGETS)
    ].copy()

    temp["Bucket"] = [
        get_bucket(t, p)
        for t, p in zip(
            temp["Target"],
            temp["Forecast"]
        )
    ]

    for target, g in temp.groupby("Target"):

        pools["global"][target] = (
            g["Residual"].to_numpy(dtype=float)
        )

    for key, g in temp.groupby(
        ["Target", "Bucket"]
    ):

        pools["group"][key] = (
            g["Residual"].to_numpy(dtype=float)
        )

    for key, g in temp.groupby(
        ["Target", "player_id"]
    ):

        pools["player"][key] = (
            g["Residual"].to_numpy(dtype=float)
        )

    return pools

# ============================================================
# 7. NORMALIZZAZIONE SICURA DEI PESI
# ============================================================

def normalize_weights(weights):

    weights = np.asarray(
        weights,
        dtype=np.float64
    )

    if len(weights) == 0:
        raise ValueError("Distribuzione vuota.")

    if not np.isfinite(weights).all():
        raise ValueError("Pesi non finiti.")

    if (weights < 0).any():
        raise ValueError("Pesi negativi.")

    total = weights.sum(dtype=np.float64)

    if not np.isfinite(total) or total <= 0:
        raise ValueError("Somma pesi non valida.")

    weights = weights / total

    if not np.isclose(
        weights.sum(),
        1.0,
        atol=TOLERANCE,
        rtol=0
    ):

        raise ValueError(
            "Normalizzazione pesi fallita."
        )

    return weights

# ============================================================
# 8. DISTRIBUZIONE PERSONALIZZATA
# ============================================================

def get_distribution(
    pools,
    target,
    player_id,
    prediction
):

    if not np.isfinite(prediction):
        raise ValueError("Forecast non valido.")

    global_errors = pools["global"].get(
        target,
        np.array([])
    )

    if len(global_errors) == 0:

        raise ValueError(
            f"Errori storici mancanti: {target}"
        )

    bucket = get_bucket(
        target,
        prediction
    )

    group_errors = pools["group"].get(
        (target, bucket),
        np.array([])
    )

    player_errors = pools["player"].get(
        (target, str(player_id)),
        np.array([])
    )

    if len(group_errors) >= MIN_GROUP_SAMPLE:

        base_errors = group_errors
        source = "Forecast Group"

    else:

        base_errors = global_errors
        source = "Global"

    # Shrinkage statistico

    if len(player_errors) >= MIN_PLAYER_SAMPLE:

        alpha = len(player_errors) / (
            len(player_errors) + 30
        )

        errors = np.concatenate([
            player_errors,
            base_errors
        ])

        weights = np.concatenate([
            np.full(
                len(player_errors),
                alpha / len(player_errors)
            ),
            np.full(
                len(base_errors),
                (1 - alpha) / len(base_errors)
            )
        ])

        source = "Player + " + source

    else:

        errors = base_errors

        weights = np.full(
            len(errors),
            1 / len(errors)
        )

    outcomes = np.maximum(
        np.floor(
            prediction + errors + 0.5
        ),
        0
    )

    if not np.isfinite(outcomes).all():
        raise ValueError("Outcome non valido.")

    order = np.argsort(outcomes)

    outcomes = outcomes[order]
    weights = weights[order]

    weights = normalize_weights(weights)

    return (
        outcomes,
        weights,
        source,
        len(player_errors),
        len(base_errors)
    )

# ============================================================
# 9. FUNZIONI STATISTICHE — FIXED
# ============================================================

def weighted_quantile(
    values,
    weights,
    q
):

    if not 0 <= q <= 1:
        raise ValueError("Quantile non valido.")

    cumulative = np.cumsum(
        weights,
        dtype=np.float64
    )

    cumulative = np.clip(
        cumulative,
        0.0,
        1.0
    )

    cumulative[-1] = 1.0

    index = np.searchsorted(
        cumulative,
        q,
        side="left"
    )

    index = min(
        index,
        len(values) - 1
    )

    return float(values[index])


def probability_at_least(
    values,
    weights,
    threshold
):

    p = float(
        weights[values >= threshold].sum(
            dtype=np.float64
        )
    )

    if not np.isfinite(p):

        raise ValueError(
            f"Probabilita non valida: {p}"
        )

    if p < -TOLERANCE or p > 1 + TOLERANCE:

        raise ValueError(
            f"Probabilita fuori intervallo: {p}"
        )

    # FIX: corregge esclusivamente
    # gli errori numerici floating point.

    return float(
        np.clip(p, 0.0, 1.0)
    )


def prediction_interval(
    values,
    weights,
    level
):

    alpha = 1 - level / 100

    lower = weighted_quantile(
        values,
        weights,
        alpha / 2
    )

    upper = weighted_quantile(
        values,
        weights,
        1 - alpha / 2
    )

    return lower, upper

# ============================================================
# 10. BACKTEST PROBABILISTICO
# ============================================================

calibration_pools = build_error_pools(
    calibration
)

coverage_rows = []
probability_rows = []

print("\nBACKTEST PROBABILISTICO...")

for r in evaluation.itertuples(index=False):

    target = r.Target

    if target not in TARGETS:
        continue

    values, weights, source, n_player, n_base = (
        get_distribution(
            calibration_pools,
            target,
            r.player_id,
            r.Forecast
        )
    )

    for level in INTERVALS:

        lower, upper = prediction_interval(
            values,
            weights,
            level
        )

        coverage_rows.append({
            "Target": target,
            "Level": level,
            "Covered": int(
                lower <= r.Actual <= upper
            ),
            "Width": upper - lower
        })

    for threshold in THRESHOLDS[target]:

        p = probability_at_least(
            values,
            weights,
            threshold
        )

        actual_event = int(
            r.Actual >= threshold
        )

        probability_rows.append({
            "Target": target,
            "Threshold": threshold,
            "Probability": p,
            "Actual_Event": actual_event,
            "Brier_Error": (
                p - actual_event
            ) ** 2
        })

coverage = pd.DataFrame(
    coverage_rows
)

coverage_report = coverage.groupby(
    ["Target", "Level"],
    as_index=False
).agg(
    Observed_Coverage=("Covered", "mean"),
    Average_Width=("Width", "mean"),
    Sample=("Covered", "size")
)

coverage_report["Observed_Coverage"] *= 100

coverage_report["Coverage_Error"] = (
    coverage_report["Observed_Coverage"]
    - coverage_report["Level"]
)

probability_backtest = pd.DataFrame(
    probability_rows
)

brier_report = probability_backtest.groupby(
    ["Target", "Threshold"],
    as_index=False
).agg(
    Brier_Score=("Brier_Error", "mean"),
    Predicted_Probability=("Probability", "mean"),
    Actual_Frequency=("Actual_Event", "mean"),
    Sample=("Actual_Event", "size")
)

print("Backtest completato.")

# ============================================================
# 11. RELIABILITY DIAGRAM
# ============================================================

probability_backtest["Probability_Bin"] = pd.cut(
    probability_backtest["Probability"],
    bins=np.linspace(0, 1, 11),
    include_lowest=True
)

reliability = probability_backtest.groupby(
    ["Target", "Probability_Bin"],
    observed=True,
    as_index=False
).agg(
    Mean_Probability=("Probability", "mean"),
    Actual_Frequency=("Actual_Event", "mean"),
    Sample=("Actual_Event", "size")
)

reliability["Calibration_Error"] = (
    reliability["Mean_Probability"]
    - reliability["Actual_Frequency"]
).abs()

reliability["Probability_Bin"] = (
    reliability["Probability_Bin"].astype(str)
)

ece_rows = []

for target, g in reliability.groupby("Target"):

    ece = np.average(
        g["Calibration_Error"],
        weights=g["Sample"]
    )

    ece_rows.append({
        "Target": target,
        "ECE_pct": 100 * ece,
        "Sample": g["Sample"].sum()
    })

ece_report = pd.DataFrame(
    ece_rows
)

# ============================================================
# 12. DISTRIBUZIONI FUTURE
# ============================================================

final_pools = build_error_pools(
    history
)

summary_rows = []
threshold_rows = []

print("\nGENERAZIONE PROBABILITA' FUTURE...")

for _, r in forecast.iterrows():

    for target in TARGETS:

        prediction = float(
            r[f"Forecast_{target}"]
        )

        if not np.isfinite(prediction):
            continue

        if prediction < 0:
            raise ValueError("Forecast negativo.")

        values, weights, source, n_player, n_base = (
            get_distribution(
                final_pools,
                target,
                r["player_id"],
                prediction
            )
        )

        expected = float(
            np.average(
                values,
                weights=weights
            )
        )

        std = float(
            np.sqrt(
                np.average(
                    (values - expected) ** 2,
                    weights=weights
                )
            )
        )

        metadata = {
            "game_id": r["game_id"],
            "player_id": r["player_id"],
            "date": r["date"].isoformat(),
            "generated_at": NOW.isoformat(),
            "Forecast_Generated_At": r.get(
                "generated_at",
                ""
            ),
            "Giocatore": r["Giocatore"],
            "Squadra": r["Squadra"],
            "Avversario": r["Avversario"],
            "Target": target,
            "Forecast": prediction,
            "Distribution_Source": source,
            "Player_Sample": n_player,
            "Reference_Sample": n_base,
            "Roster_Status": r.get(
                "Roster_Status",
                "Disponibilita non verificata"
            ),
            "Probability_Version": "3.1"
        }

        summary = {
            **metadata,
            "Expected_Value": expected,
            "Median": weighted_quantile(
                values, weights, 0.50
            ),
            "Std_Dev": std,
            "P10": weighted_quantile(
                values, weights, 0.10
            ),
            "P90": weighted_quantile(
                values, weights, 0.90
            )
        }

        for level in INTERVALS:

            lower, upper = prediction_interval(
                values,
                weights,
                level
            )

            summary[f"Lower_{level}"] = lower
            summary[f"Upper_{level}"] = upper

        summary_rows.append(summary)

        for threshold in THRESHOLDS[target]:

            p = probability_at_least(
                values,
                weights,
                threshold
            )

            threshold_rows.append({
                **metadata,
                "Threshold": threshold,
                "Probability_At_Least": p,
                "Probability_Below": 1 - p,
                "Probability_pct": 100 * p
            })

summary_df = pd.DataFrame(
    summary_rows
)

probabilities = pd.DataFrame(
    threshold_rows
)

if summary_df.empty or probabilities.empty:

    raise ValueError(
        "Nessuna distribuzione valida."
    )

# ============================================================
# 13. CONTROLLO QUALITA' — FIXED
# ============================================================

print("\nCONTROLLO QUALITA'...")

p = probabilities[
    "Probability_At_Least"
].to_numpy(dtype=float)

if not np.isfinite(p).all():

    raise ValueError(
        "Probabilita NaN o infinite."
    )

if (
    (p < -TOLERANCE)
    | (p > 1 + TOLERANCE)
).any():

    raise ValueError(
        "Probabilita realmente fuori intervallo."
    )

# Correzione finale floating point.

probabilities["Probability_At_Least"] = (
    np.clip(p, 0.0, 1.0)
)

probabilities["Probability_Below"] = (
    1 - probabilities["Probability_At_Least"]
)

probabilities["Probability_pct"] = (
    100 * probabilities["Probability_At_Least"]
)

assert probabilities[
    "Probability_At_Least"
].between(0, 1).all()

assert np.allclose(
    probabilities["Probability_At_Least"]
    + probabilities["Probability_Below"],
    1.0,
    atol=TOLERANCE,
    rtol=0
)

# Controllo intervalli predittivi.

for level in INTERVALS:

    assert (
        summary_df[f"Lower_{level}"]
        <= summary_df[f"Upper_{level}"]
    ).all()

for small, large in zip(
    INTERVALS[:-1],
    INTERVALS[1:]
):

    assert (
        summary_df[f"Lower_{large}"]
        <= summary_df[f"Lower_{small}"]
    ).all()

    assert (
        summary_df[f"Upper_{large}"]
        >= summary_df[f"Upper_{small}"]
    ).all()

# Controllo monotonicita:
# P(X >= 30) non puo superare P(X >= 20).

probabilities = probabilities.sort_values(
    [
        "game_id",
        "player_id",
        "Target",
        "Threshold"
    ]
).reset_index(drop=True)

diff = probabilities.groupby(
    ["game_id", "player_id", "Target"]
)["Probability_At_Least"].diff()

if (diff > TOLERANCE).any():

    raise ValueError(
        "Errore monotonicita probabilita."
    )

# Rimuove eventuali partite iniziate
# durante il calcolo.

FINISHED_AT = pd.Timestamp.now(tz="UTC")

summary_df = summary_df[
    pd.to_datetime(
        summary_df["date"],
        utc=True
    ) > FINISHED_AT
].copy()

probabilities = probabilities[
    pd.to_datetime(
        probabilities["date"],
        utc=True
    ) > FINISHED_AT
].copy()

if probabilities.empty or summary_df.empty:

    raise ValueError(
        "Forecast scaduti durante il calcolo."
    )

print("Probabilita min:", probabilities[
    "Probability_At_Least"
].min())

print("Probabilita max:", probabilities[
    "Probability_At_Least"
].max())

print("Controllo intervalli: OK")
print("Controllo monotonicita: OK")
print("CONTROLLO QUALITA' SUPERATO!")

# ============================================================
# 14. ARCHIVIAZIONE PERMANENTE SQLITE
# ============================================================

with sqlite3.connect(
    DB_PATH,
    timeout=60
) as conn:

    conn.execute("""
        CREATE TABLE IF NOT EXISTS
        nba_probability_history_v3 (
            game_id TEXT NOT NULL,
            player_id TEXT NOT NULL,
            generated_at TEXT NOT NULL,
            game_date TEXT NOT NULL,
            player_name TEXT,
            target TEXT NOT NULL,
            threshold REAL NOT NULL,
            forecast REAL,
            probability REAL,
            distribution_source TEXT,
            PRIMARY KEY (
                game_id,
                player_id,
                generated_at,
                target,
                threshold
            )
        )
    """)

    records = []

    for r in probabilities.itertuples(index=False):

        records.append((
            str(r.game_id),
            str(r.player_id),
            r.generated_at,
            r.date,
            r.Giocatore,
            r.Target,
            float(r.Threshold),
            float(r.Forecast),
            float(r.Probability_At_Least),
            r.Distribution_Source
        ))

    conn.executemany("""
        INSERT OR IGNORE INTO
        nba_probability_history_v3 (
            game_id,
            player_id,
            generated_at,
            game_date,
            player_name,
            target,
            threshold,
            forecast,
            probability,
            distribution_source
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, records)

    total_history = conn.execute("""
        SELECT COUNT(*)
        FROM nba_probability_history_v3
    """).fetchone()[0]

print("\nARCHIVIAZIONE SQLITE COMPLETATA")

# ============================================================
# 15. ESPORTAZIONE FILE
# ============================================================

stamp = NOW.strftime(
    "%Y%m%d_%H%M%S_%f"
)

archive_path = os.path.join(
    ARCHIVE_DIR,
    f"NBA_Probabilities_{stamp}.csv"
)

summary_archive_path = os.path.join(
    ARCHIVE_DIR,
    f"NBA_Distributions_{stamp}.csv"
)

probabilities.to_csv(
    archive_path,
    index=False,
    encoding="utf-8-sig"
)

summary_df.to_csv(
    summary_archive_path,
    index=False,
    encoding="utf-8-sig"
)

probabilities.to_csv(
    CSV_PATH + ".tmp",
    index=False,
    encoding="utf-8-sig"
)

summary_df.to_csv(
    SUMMARY_PATH + ".tmp",
    index=False,
    encoding="utf-8-sig"
)

with pd.ExcelWriter(
    OUTPUT_PATH + ".tmp.xlsx",
    engine="openpyxl"
) as writer:

    sheets = {
        "Player Distributions": summary_df,
        "Threshold Probabilities": probabilities,
        "Interval Coverage": coverage_report,
        "Brier Score": brier_report,
        "Reliability": reliability,
        "Calibration ECE": ece_report
    }

    for name, data in sheets.items():

        data.to_excel(
            writer,
            sheet_name=name,
            index=False
        )

        sheet = writer.sheets[name]

        sheet.freeze_panes = "A2"

        sheet.auto_filter.ref = (
            sheet.dimensions
        )

os.replace(
    CSV_PATH + ".tmp",
    CSV_PATH
)

os.replace(
    SUMMARY_PATH + ".tmp",
    SUMMARY_PATH
)

os.replace(
    OUTPUT_PATH + ".tmp.xlsx",
    OUTPUT_PATH
)

print("FILE SALVATI CORRETTAMENTE")

# ============================================================
# 16. ANALISI CALIBRAZIONE
# ============================================================

print("\nINTERVAL COVERAGE")

display(
    coverage_report.round(3)
)

print("\nEXPECTED CALIBRATION ERROR")

display(
    ece_report.round(3)
)

print("\nBRIER SCORE")

display(
    brier_report.round(4)
)

fig = px.scatter(
    reliability,
    x="Mean_Probability",
    y="Actual_Frequency",
    color="Target",
    size="Sample",
    title="NBA V3.1 — Probability Calibration",
    template="plotly_dark"
)

fig.add_shape(
    type="line",
    x0=0,
    y0=0,
    x1=1,
    y1=1,
    line=dict(
        color="white",
        dash="dash"
    )
)

fig.update_xaxes(
    range=[0, 1],
    tickformat=".0%"
)

fig.update_yaxes(
    range=[0, 1],
    tickformat=".0%"
)

fig.show()

fig = px.bar(
    coverage_report,
    x="Level",
    y="Observed_Coverage",
    color="Target",
    barmode="group",
    title="NBA V3.1 — Prediction Interval Coverage",
    template="plotly_dark"
)

fig.update_yaxes(
    range=[0, 100],
    ticksuffix="%"
)

fig.show()

fig = px.bar(
    ece_report,
    x="Target",
    y="ECE_pct",
    color="Target",
    title="NBA V3.1 — Calibration Error",
    template="plotly_dark"
)

fig.update_yaxes(
    ticksuffix="%"
)

fig.show()

# ============================================================
# 17. PLAYER PROBABILITY EXPLORER
# ============================================================

player_options = (
    summary_df[["player_id", "Giocatore"]]
    .drop_duplicates("player_id")
    .sort_values("Giocatore")
)

players = [
    (
        f"{r.Giocatore} ({r.player_id})",
        r.player_id
    )
    for r in player_options.itertuples(index=False)
]

default_player = next(
    (
        pid for name, pid in players
        if "jokic" in name.lower()
        or "jokić" in name.lower()
    ),
    players[0][1]
)

player_widget = widgets.Dropdown(
    options=players,
    value=default_player,
    description="Player:"
)

target_widget = widgets.Dropdown(
    options=TARGETS,
    value="PTS",
    description="Stat:"
)

game_widget = widgets.Dropdown(
    options=[],
    description="Game:"
)

threshold_widget = widgets.IntSlider(
    value=20,
    min=0,
    max=100,
    step=1,
    description="Soglia:",
    continuous_update=False
)

output = widgets.Output()

def update_games():

    selected = summary_df[
        summary_df["player_id"]
        == player_widget.value
    ].drop_duplicates(
        "game_id"
    ).sort_values("date")

    options = []

    for r in selected.itertuples(index=False):

        date_it = pd.Timestamp(
            r.date
        ).tz_convert(
            "Europe/Rome"
        ).strftime(
            "%d/%m %H:%M"
        )

        label = (
            f"{date_it} — vs {r.Avversario}"
        )

        options.append(
            (label, r.game_id)
        )

    game_widget.options = options

    if options:
        game_widget.value = options[0][1]


def show_player(change=None):

    with output:

        clear_output(wait=True)

        player_id = player_widget.value
        target = target_widget.value
        game_id = game_widget.value

        selected = summary_df[
            (summary_df["player_id"] == player_id)
            & (summary_df["Target"] == target)
            & (summary_df["game_id"] == game_id)
        ]

        if selected.empty:

            print("Nessuna previsione.")
            return

        r = selected.iloc[0]

        selected_probs = probabilities[
            (probabilities["player_id"] == player_id)
            & (probabilities["game_id"] == game_id)
            & (probabilities["Target"] == target)
        ].sort_values("Threshold")

        values, weights, _, _, _ = (
            get_distribution(
                final_pools,
                target,
                player_id,
                r["Forecast"]
            )
        )

        custom_threshold = (
            threshold_widget.value
        )

        custom_probability = (
            probability_at_least(
                values,
                weights,
                custom_threshold
            )
        )

        print("=" * 60)

        print(
            r["Giocatore"].upper()
        )

        print(
            r["Squadra"],
            "vs",
            r["Avversario"]
        )

        print("Target:", target)

        print(
            "Forecast:",
            round(r["Forecast"], 2)
        )

        print(
            "Expected Value:",
            round(r["Expected_Value"], 2)
        )

        print(
            "Mediana:",
            r["Median"]
        )

        print(
            "Deviazione standard:",
            round(r["Std_Dev"], 2)
        )

        print(
            "Campione individuale:",
            r["Player_Sample"]
        )

        print(
            "Campione riferimento:",
            r["Reference_Sample"]
        )

        print(
            "Distribuzione:",
            r["Distribution_Source"]
        )

        print(
            "Disponibilita:",
            r["Roster_Status"]
        )

        print("=" * 60)

        print("\nSOGLIA PERSONALIZZATA")

        print(
            f"P({target} >= {custom_threshold}): "
            f"{100 * custom_probability:.2f}%"
        )

        print(
            f"P({target} < {custom_threshold}): "
            f"{100 * (1-custom_probability):.2f}%"
        )

        print("\nINTERVALLI PREDITTIVI")

        for level in INTERVALS:

            print(
                f"{level}%:",
                r[f"Lower_{level}"],
                "-",
                r[f"Upper_{level}"]
            )

        print("\nPROBABILITA' PER SOGLIA")

        display(
            selected_probs[[
                "Threshold",
                "Probability_pct"
            ]].round(2)
        )

        distribution = pd.DataFrame({
            "Outcome": values,
            "Probability": weights
        }).groupby(
            "Outcome",
            as_index=False
        ).sum()

        distribution["Threshold_Status"] = np.where(
            distribution["Outcome"] >= custom_threshold,
            "Soglia raggiunta",
            "Sotto soglia"
        )

        fig = px.bar(
            distribution,
            x="Outcome",
            y="Probability",
            color="Threshold_Status",
            title=(
                f"{r['Giocatore']} — "
                f"{target} Distribution"
            ),
            template="plotly_dark",
            color_discrete_map={
                "Soglia raggiunta": "#22c55e",
                "Sotto soglia": "#64748b"
            }
        )

        fig.add_vline(
            x=r["Forecast"],
            line_dash="dash",
            line_color="orange",
            annotation_text="Forecast"
        )

        fig.add_vline(
            x=custom_threshold,
            line_dash="dot",
            line_color="white",
            annotation_text="Soglia"
        )

        fig.update_yaxes(
            tickformat=".0%"
        )

        fig.show()

        fig = px.bar(
            selected_probs,
            x="Threshold",
            y="Probability_pct",
            color="Probability_pct",
            title=(
                f"{r['Giocatore']} — "
                f"{target} Threshold Probabilities"
            ),
            template="plotly_dark"
        )

        fig.update_yaxes(
            range=[0, 100],
            ticksuffix="%"
        )

        fig.show()


def on_player_change(change):

    update_games()
    show_player()


def on_target_change(change):

    defaults = {
        "PTS": 20,
        "REB": 8,
        "AST": 5,
        "3PM": 2,
        "PRA": 30
    }

    threshold_widget.value = (
        defaults[target_widget.value]
    )

    show_player()


update_games()

player_widget.observe(
    on_player_change,
    names="value"
)

target_widget.observe(
    on_target_change,
    names="value"
)

game_widget.observe(
    show_player,
    names="value"
)

threshold_widget.observe(
    show_player,
    names="value"
)

display(
    widgets.VBox([
        widgets.HBox([
            player_widget,
            target_widget
        ]),
        game_widget,
        threshold_widget,
        output
    ])
)

show_player()

# ============================================================
# 18. RIEPILOGO FINALE
# ============================================================

print("\n" + "=" * 60)

print("STEP 18 COMPLETATO!")
print("PROBABILITY ENGINE V3.1")

print("=" * 60)

print(
    "Errori storici:",
    len(history)
)

print(
    "Partite future:",
    summary_df["game_id"].nunique()
)

print(
    "Giocatori:",
    summary_df["player_id"].nunique()
)

print(
    "Distribuzioni generate:",
    len(summary_df)
)

print(
    "Probabilita calcolate:",
    len(probabilities)
)

print(
    "Probabilita archiviate:",
    total_history
)

print("\nFILE SALVATI")

print(OUTPUT_PATH)
print(CSV_PATH)
print(SUMMARY_PATH)
print(archive_path)
print(summary_archive_path)

print("\nTUTTI I CONTROLLI SUPERATI")

print("\nPRONTO PER STEP 19!")


In [ ]:
# Injury and lineup context
# Source: notebook cell 47 — Injury and lineup context
# NOTE: Colab-oriented notebook code; requires runtime/Drive configuration.
# Not verified as standalone executable.


# ============================================================
# STEP 19 — INJURY & LINEUP INTELLIGENCE
# NBA DATA SCIENCE ANALYTICS
# VERSION 3.2
# ============================================================

import os
import sqlite3
import requests
import numpy as np
import pandas as pd
import plotly.express as px
import ipywidgets as widgets

from google.colab import drive
from IPython.display import display, clear_output
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ============================================================
# 1. RIPRISTINO AUTOMATICO AMBIENTE
# ============================================================

# Preserve the directory selected in the setup cell.
BASE_DIR = globals().get("BASE_DIR") or os.environ.get("NBA_DATA_DIR")
if not BASE_DIR or not os.path.isfile(os.path.join(BASE_DIR, "nba_database.db")):
    raise FileNotFoundError("Run the setup cell and collect the NBA database first.")


NOW = pd.Timestamp.now(tz="UTC")

DB_PATH = os.path.join(
    BASE_DIR, "nba_database.db"
)

FORECAST_PATH = os.path.join(
    BASE_DIR, "NBA_Forecast_Engine_V3.csv"
)

PROBABILITY_PATH = os.path.join(
    BASE_DIR, "NBA_Probability_Engine_V3.csv"
)

OUTPUT_PATH = os.path.join(
    BASE_DIR, "NBA_Context_Aware_Forecast.xlsx"
)

CSV_PATH = os.path.join(
    BASE_DIR, "NBA_Context_Aware_Forecast.csv"
)

INJURY_PATH = os.path.join(
    BASE_DIR, "NBA_Injury_Report.csv"
)

ARCHIVE_DIR = os.path.join(
    BASE_DIR, "Injury_Archive"
)

os.makedirs(ARCHIVE_DIR, exist_ok=True)

print("Cartella progetto:", BASE_DIR)

# ============================================================
# 2. CONFIGURAZIONE
# ============================================================

TARGETS = {
    "PTS": "pts",
    "REB": "reb",
    "AST": "ast",
    "3PM": "fg3m",
    "PRA": "PRA"
}

MIN_IMPACT_SAMPLE = 5

HISTORY_DAYS = 365

ESPN_INJURIES_URL = (
    "https://site.api.espn.com/apis/site/v2/"
    "sports/basketball/nba/injuries"
)

session = requests.Session()

retry = Retry(
    total=3,
    backoff_factor=1,
    status_forcelist=[
        429, 500, 502, 503, 504
    ],
    allowed_methods=["GET"]
)

session.mount(
    "https://",
    HTTPAdapter(max_retries=retry)
)

session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
})

# ============================================================
# 3. CARICAMENTO FORECAST
# ============================================================

if not os.path.isfile(FORECAST_PATH):

    raise FileNotFoundError(
        "Completa prima STEP 17."
    )

forecast = pd.read_csv(
    FORECAST_PATH,
    dtype={
        "game_id": str,
        "player_id": str
    }
)

forecast["date"] = pd.to_datetime(
    forecast["date"],
    utc=True,
    errors="coerce",
    format="mixed"
)

forecast = forecast[
    forecast["date"] > NOW
].drop_duplicates(
    ["game_id", "player_id"]
).copy()

if forecast.empty:

    raise ValueError(
        "Forecast scaduti. Riesegui STEP 17."
    )

print("Forecast futuri:", len(forecast))

# ============================================================
# 4. DOWNLOAD INJURY REPORT ESPN
# ============================================================

def normalize_status(value):

    value = str(value).strip().lower()

    mapping = {
        "out": "Out",
        "doubtful": "Doubtful",
        "questionable": "Questionable",
        "day-to-day": "Questionable",
        "day to day": "Questionable",
        "probable": "Probable",
        "available": "Available",
        "active": "Available",
        "game time decision": "Questionable",
        "gtd": "Questionable"
    }

    return mapping.get(value, "Unknown")


injury_rows = []

injury_feed_ok = False

print("\nDOWNLOAD INJURY REPORT ESPN...")

try:

    response = session.get(
        ESPN_INJURIES_URL,
        timeout=30
    )

    response.raise_for_status()

    payload = response.json()

    groups = payload.get("injuries")

    if not isinstance(groups, list):

        raise ValueError(
            "Struttura ESPN non riconosciuta."
        )

    for group in groups:

        team = group.get("displayName", "")

        team_info = group.get("team", {})

        if isinstance(team_info, dict):

            team = team_info.get(
                "displayName", team
            )

        items = group.get("injuries", [])

        if not isinstance(items, list):
            continue

        for injury in items:

            athlete = injury.get("athlete", {})

            if not isinstance(athlete, dict):
                continue

            player_id = athlete.get("id")

            if player_id is None:
                continue

            details = injury.get("details") or {}

            if not isinstance(details, dict):
                details = {}

            raw_status = injury.get("status", "")

            if isinstance(raw_status, dict):

                raw_status = raw_status.get(
                    "name",
                    raw_status.get("description", "")
                )

            status = normalize_status(
                raw_status
            )

            injury_rows.append({
                "player_id": str(player_id),
                "Giocatore": athlete.get(
                    "displayName",
                    athlete.get("fullName", "")
                ),
                "Squadra_ESPN": team,
                "Injury_Status": status,
                "Raw_Status": raw_status,
                "Injury_Type": details.get(
                    "type", ""
                ),
                "Injury_Location": details.get(
                    "location", ""
                ),
                "Return_Date": details.get(
                    "returnDate", ""
                ),
                "Comment": injury.get(
                    "longComment",
                    injury.get("shortComment", "")
                ),
                "Report_Date": injury.get(
                    "date", ""
                ),
                "Fetched_At": NOW.isoformat(),
                "Source": ESPN_INJURIES_URL
            })

    injury_feed_ok = True

except Exception as e:

    print("ATTENZIONE: injury feed non disponibile.")
    print(type(e).__name__, str(e))

injury_columns = [
    "player_id",
    "Giocatore",
    "Squadra_ESPN",
    "Injury_Status",
    "Raw_Status",
    "Injury_Type",
    "Injury_Location",
    "Return_Date",
    "Comment",
    "Report_Date",
    "Fetched_At",
    "Source"
]

injuries = pd.DataFrame(
    injury_rows,
    columns=injury_columns
)

if not injuries.empty:

    injuries["_report_time"] = pd.to_datetime(
        injuries["Report_Date"],
        utc=True,
        errors="coerce",
        format="mixed"
    )

    injuries = injuries.sort_values(
        "_report_time",
        na_position="first"
    ).drop_duplicates(
        "player_id",
        keep="last"
    ).drop(
        columns="_report_time"
    )

print("Feed disponibile:", injury_feed_ok)
print("Infortuni identificati:", len(injuries))

# ============================================================
# 5. CARICAMENTO DATABASE STORICO
# ============================================================

with sqlite3.connect(DB_PATH) as conn:

    box = pd.read_sql_query(
        "SELECT * FROM espn_player_boxscores",
        conn
    )

    games = pd.read_sql_query(
        "SELECT * FROM espn_games",
        conn
    )

for data in [box, games]:

    data["game_id"] = (
        data["game_id"].astype(str)
    )

    data["game_date"] = pd.to_datetime(
        data["game_date"],
        utc=True,
        errors="coerce",
        format="mixed"
    )

for col in ["player_id", "team_id"]:

    box[col] = box[col].astype(str)

numeric_columns = [
    "minutes",
    "pts",
    "reb",
    "ast",
    "fg3m",
    "starter",
    "did_not_play"
]

for col in numeric_columns:

    box[col] = pd.to_numeric(
        box[col],
        errors="coerce"
    )

games["completed"] = pd.to_numeric(
    games["completed"],
    errors="coerce"
)

completed_ids = set(
    games.loc[
        (games["completed"] == 1)
        & (games["game_date"] < NOW),
        "game_id"
    ]
)

box = box[
    box["game_id"].isin(completed_ids)
    & (box["game_date"] < NOW)
    & (
        box["game_date"]
        >= NOW - pd.Timedelta(days=HISTORY_DAYS)
    )
].drop_duplicates(
    ["game_id", "player_id"]
).sort_values(
    "game_date"
).copy()

box["PRA"] = (
    box["pts"] + box["reb"] + box["ast"]
)

played = box[
    (box["minutes"] > 0)
    & (box["did_not_play"] == 0)
].copy()

if played.empty:

    raise ValueError(
        "Nessuna presenza storica valida."
    )

print("\nPresenze storiche:", len(played))

# ============================================================
# 6. ANALISI ROTAZIONI E MINUTI
# ============================================================

rotation_rows = []

for player_id, group in played.groupby(
    "player_id"
):

    group = group.sort_values(
        "game_date"
    )

    last = group.iloc[-1]

    # Utilizziamo la squadra piu recente
    # per evitare di mescolare rotazioni
    # precedenti a un eventuale trasferimento.

    current_team = group[
        group["team_id"] == last["team_id"]
    ]

    last5 = current_team.tail(5)
    last10 = current_team.tail(10)
    last20 = current_team.tail(20)

    minutes5 = last5["minutes"].mean()
    minutes10 = last10["minutes"].mean()
    minutes20 = last20["minutes"].mean()

    starter_rate = last10["starter"].mean()

    if pd.isna(starter_rate):

        role = "Unknown"

    elif starter_rate >= 0.7:

        role = "Likely Starter"

    elif starter_rate >= 0.3:

        role = "Rotation / Starter"

    else:

        role = "Bench"

    rotation_rows.append({
        "player_id": player_id,
        "team_id": last["team_id"],
        "Last_Team": last["team_name"],
        "Last_Appearance": (
            last["game_date"].isoformat()
        ),
        "Minutes_Last5": minutes5,
        "Minutes_Last10": minutes10,
        "Minutes_Last20": minutes20,
        "Minutes_Trend": minutes5 - minutes20,
        "Starter_Rate_Last10": starter_rate,
        "Estimated_Role": role,
        "Rotation_Sample": len(last10),
        "PTS_Last10": last10["pts"].mean(),
        "REB_Last10": last10["reb"].mean(),
        "AST_Last10": last10["ast"].mean()
    })

rotations = pd.DataFrame(rotation_rows)

print("Profili rotazione:", len(rotations))

# ============================================================
# 7. INTEGRAZIONE INJURY REPORT E FORECAST
# ============================================================

injury_fields = [
    "player_id",
    "Injury_Status",
    "Raw_Status",
    "Injury_Type",
    "Injury_Location",
    "Return_Date",
    "Comment",
    "Report_Date",
    "Fetched_At"
]

context = forecast.merge(
    injuries[injury_fields],
    on="player_id",
    how="left",
    validate="many_to_one"
)

context = context.merge(
    rotations,
    on="player_id",
    how="left",
    validate="many_to_one"
)

context["Injury_Status"] = (
    context["Injury_Status"].fillna("Unknown")
)

context["Injury_Feed_OK"] = injury_feed_ok

context["Listed_In_Injury_Feed"] = (
    context["player_id"].isin(
        injuries["player_id"]
    )
)

def availability_flag(row):

    status = row["Injury_Status"]

    if status == "Out":
        return "EXCLUDED"

    if status in ["Doubtful", "Questionable"]:
        return "REVIEW_REQUIRED"

    if status == "Probable":
        return "PROBABLE"

    if status == "Available":
        return "AVAILABLE"

    if not row["Injury_Feed_OK"]:
        return "FEED_UNAVAILABLE"

    return "NOT_VERIFIED"


context["Availability_Flag"] = context.apply(
    availability_flag,
    axis=1
)

# ============================================================
# 8. ANALISI IMPATTO ASSENZE COMPAGNI
# ============================================================

# Confronto descrittivo delle prestazioni
# con compagno presente e con compagno
# esplicitamente registrato come DNP.
#
# Non assumiamo che un giocatore assente
# dal boxscore fosse necessariamente infortunato.

impact_rows = []

out_players = context[
    context["Injury_Status"] == "Out"
][[
    "game_id",
    "player_id",
    "Giocatore",
    "team_id"
]].dropna(
    subset=["team_id"]
).drop_duplicates()

for absent in out_players.itertuples(
    index=False
):

    absent_history = box[
        (box["player_id"] == absent.player_id)
        & (box["team_id"] == absent.team_id)
    ]

    games_with = set(
        absent_history.loc[
            (absent_history["minutes"] > 0)
            & (absent_history["did_not_play"] == 0),
            "game_id"
        ]
    )

    games_without = set(
        absent_history.loc[
            absent_history["did_not_play"] == 1,
            "game_id"
        ]
    )

    games_without -= games_with

    if min(
        len(games_with),
        len(games_without)
    ) < MIN_IMPACT_SAMPLE:

        continue

    teammates = context[
        (context["game_id"] == absent.game_id)
        & (context["team_id"] == absent.team_id)
        & (context["player_id"] != absent.player_id)
        & (context["Injury_Status"] != "Out")
    ]

    for player in teammates.itertuples(
        index=False
    ):

        hist = played[
            (played["player_id"] == player.player_id)
            & (played["team_id"] == absent.team_id)
        ]

        with_player = hist[
            hist["game_id"].isin(games_with)
        ]

        without_player = hist[
            hist["game_id"].isin(games_without)
        ]

        if min(
            len(with_player),
            len(without_player)
        ) < MIN_IMPACT_SAMPLE:

            continue

        for target, col in {
            **TARGETS,
            "MIN": "minutes"
        }.items():

            with_values = (
                with_player[col].dropna()
            )

            without_values = (
                without_player[col].dropna()
            )

            if min(
                len(with_values),
                len(without_values)
            ) < MIN_IMPACT_SAMPLE:

                continue

            mean_with = with_values.mean()

            mean_without = (
                without_values.mean()
            )

            delta = mean_without - mean_with

            delta_pct = (
                100 * delta / mean_with
                if mean_with > 0
                else np.nan
            )

            impact_rows.append({
                "game_id": absent.game_id,
                "player_id": player.player_id,
                "Giocatore": player.Giocatore,
                "Absent_Player_ID": absent.player_id,
                "Absent_Player": absent.Giocatore,
                "Target": target,
                "Games_With": len(with_values),
                "Games_Without": len(without_values),
                "Mean_With": mean_with,
                "Mean_Without": mean_without,
                "Historical_Delta": delta,
                "Historical_Delta_pct": delta_pct
            })

impact_columns = [
    "game_id",
    "player_id",
    "Giocatore",
    "Absent_Player_ID",
    "Absent_Player",
    "Target",
    "Games_With",
    "Games_Without",
    "Mean_With",
    "Mean_Without",
    "Historical_Delta",
    "Historical_Delta_pct"
]

impact = pd.DataFrame(
    impact_rows,
    columns=impact_columns
)

print("\nTEammate Impact Analysis")
print("Confronti disponibili:", len(impact))

# ============================================================
# 9. CONTEXT-AWARE FORECAST
# ============================================================

# Non modifichiamo arbitrariamente
# le previsioni ML in base alle assenze.
#
# Manteniamo i forecast originali e
# rimuoviamo quelli dei giocatori Out
# dalla lista delle previsioni utilizzabili.

for target in TARGETS:

    context[f"Context_{target}"] = (
        context[f"Forecast_{target}"]
    )

    context.loc[
        context["Injury_Status"] == "Out",
        f"Context_{target}"
    ] = np.nan

context["Context_Generated_At"] = (
    NOW.isoformat()
)

context["Context_Version"] = "3.2"

context["Forecast_Condition"] = (
    "Condizionato alla partecipazione"
)

context["Impact_Adjustment_Applied"] = False

context["Confirmed_Lineup"] = False

# Conteggio compagni indisponibili.

out_counts = context[
    context["Injury_Status"] == "Out"
].groupby(
    ["game_id", "Squadra"]
)["player_id"].nunique().rename(
    "Listed_Out_Teammates"
).reset_index()

context = context.merge(
    out_counts,
    on=["game_id", "Squadra"],
    how="left"
)

context["Listed_Out_Teammates"] = (
    context["Listed_Out_Teammates"]
    .fillna(0)
    .astype(int)
)

context.loc[
    context["Injury_Status"] == "Out",
    "Listed_Out_Teammates"
] -= 1

context["Listed_Out_Teammates"] = (
    context["Listed_Out_Teammates"].clip(lower=0)
)

if not injury_feed_ok:

    context["Listed_Out_Teammates"] = np.nan

# ============================================================
# 10. INTEGRAZIONE PROBABILITA STEP 18
# ============================================================

probability_context = pd.DataFrame()

if os.path.isfile(PROBABILITY_PATH):

    probabilities = pd.read_csv(
        PROBABILITY_PATH,
        dtype={
            "game_id": str,
            "player_id": str
        }
    )

    metadata = context[[
        "game_id",
        "player_id",
        "generated_at",
        "Injury_Status",
        "Availability_Flag",
        "Estimated_Role",
        "Listed_Out_Teammates"
    ]].rename(columns={
        "generated_at": "Current_Forecast_Generated_At"
    })

    probability_context = probabilities.merge(
        metadata,
        on=["game_id", "player_id"],
        how="inner",
        validate="many_to_one"
    )

    # Verifica che le probabilita derivino
    # dalla stessa versione del forecast.

    if "Forecast_Generated_At" in (
        probability_context.columns
    ):

        source_time = pd.to_datetime(
            probability_context[
                "Forecast_Generated_At"
            ],
            utc=True,
            errors="coerce",
            format="mixed"
        )

        current_time = pd.to_datetime(
            probability_context[
                "Current_Forecast_Generated_At"
            ],
            utc=True,
            errors="coerce",
            format="mixed"
        )

        probability_context = (
            probability_context[
                source_time.notna()
                & (source_time == current_time)
            ].copy()
        )

    else:

        print(
            "Probabilita senza timestamp sorgente: "
            "riesegui STEP 18 versione 3.1."
        )

        probability_context = pd.DataFrame()

    if not probability_context.empty:

        probability_context[
            "Conditional_Probability"
        ] = probability_context[
            "Probability_At_Least"
        ]

        probability_context.loc[
            probability_context["Injury_Status"] == "Out",
            "Conditional_Probability"
        ] = np.nan

        probability_context[
            "Conditional_Probability_pct"
        ] = (
            100
            * probability_context[
                "Conditional_Probability"
            ]
        )

        probability_context[
            "Participation_Probability"
        ] = np.nan

        probability_context[
            "Probability_Condition"
        ] = "Se il giocatore partecipa"

print(
    "\nProbabilita integrate:",
    len(probability_context)
)

# ============================================================
# 11. ANALISI QUINTETTI STIMATI
# ============================================================

lineup = context[
    context["Injury_Status"] != "Out"
].copy()

lineup = lineup.sort_values(
    [
        "game_id",
        "Squadra",
        "Starter_Rate_Last10",
        "Minutes_Last10"
    ],
    ascending=[
        True, True, False, False
    ]
)

lineup = lineup.groupby(
    ["game_id", "Squadra"],
    group_keys=False
).head(5).copy()

lineup["Lineup_Status"] = (
    "Proiezione storica non confermata"
)

lineup_columns = [
    "game_id",
    "date",
    "Squadra",
    "Avversario",
    "Giocatore",
    "Injury_Status",
    "Estimated_Role",
    "Starter_Rate_Last10",
    "Minutes_Last10",
    "Lineup_Status"
]

lineup = lineup[lineup_columns]

# ============================================================
# 12. CONTROLLO QUALITA'
# ============================================================

if context.duplicated(
    ["game_id", "player_id"]
).any():

    raise ValueError(
        "Forecast duplicati."
    )

out_mask = (
    context["Injury_Status"] == "Out"
)

for target in TARGETS:

    if not context.loc[
        out_mask,
        f"Context_{target}"
    ].isna().all():

        raise ValueError(
            "Giocatore Out non escluso."
        )

if not probability_context.empty:

    valid_p = probability_context[
        "Conditional_Probability"
    ].dropna()

    if not valid_p.between(0, 1).all():

        raise ValueError(
            "Probabilita non valide."
        )

FINISHED_AT = pd.Timestamp.now(tz="UTC")

context = context[
    context["date"] > FINISHED_AT
].copy()

valid_games = set(context["game_id"])

lineup = lineup[
    lineup["game_id"].isin(valid_games)
].copy()

impact = impact[
    impact["game_id"].isin(valid_games)
].copy()

if not probability_context.empty:

    probability_context = (
        probability_context[
            probability_context["game_id"]
            .isin(valid_games)
        ].copy()
    )

if context.empty:

    raise ValueError(
        "Nessun forecast futuro valido."
    )

print("\nCONTROLLO QUALITA' SUPERATO!")

# ============================================================
# 13. ARCHIVIAZIONE SQLITE
# ============================================================

with sqlite3.connect(
    DB_PATH,
    timeout=60
) as conn:

    conn.execute("""
        CREATE TABLE IF NOT EXISTS
        nba_injury_snapshots (
            player_id TEXT NOT NULL,
            fetched_at TEXT NOT NULL,
            player_name TEXT,
            team_name TEXT,
            injury_status TEXT,
            raw_status TEXT,
            injury_type TEXT,
            injury_location TEXT,
            report_date TEXT,
            source TEXT,
            PRIMARY KEY (
                player_id,
                fetched_at
            )
        )
    """)

    records = []

    for r in injuries.itertuples(
        index=False
    ):

        records.append((
            r.player_id,
            r.Fetched_At,
            r.Giocatore,
            r.Squadra_ESPN,
            r.Injury_Status,
            str(r.Raw_Status),
            str(r.Injury_Type),
            str(r.Injury_Location),
            str(r.Report_Date),
            r.Source
        ))

    conn.executemany("""
        INSERT OR IGNORE INTO
        nba_injury_snapshots
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, records)

    conn.execute("""
        CREATE TABLE IF NOT EXISTS
        nba_injury_feed_runs (
            fetched_at TEXT PRIMARY KEY,
            feed_ok INTEGER,
            injury_count INTEGER,
            source TEXT
        )
    """)

    conn.execute("""
        INSERT OR IGNORE INTO
        nba_injury_feed_runs
        VALUES (?, ?, ?, ?)
    """, (
        NOW.isoformat(),
        int(injury_feed_ok),
        len(injuries),
        ESPN_INJURIES_URL
    ))

    conn.execute("""
        CREATE TABLE IF NOT EXISTS
        nba_availability_history (
            game_id TEXT NOT NULL,
            player_id TEXT NOT NULL,
            generated_at TEXT NOT NULL,
            game_date TEXT,
            player_name TEXT,
            team_name TEXT,
            injury_status TEXT,
            availability_flag TEXT,
            feed_ok INTEGER,
            PRIMARY KEY (
                game_id,
                player_id,
                generated_at
            )
        )
    """)

    availability_records = []

    for r in context.itertuples(
        index=False
    ):

        availability_records.append((
            r.game_id,
            r.player_id,
            NOW.isoformat(),
            r.date.isoformat(),
            r.Giocatore,
            r.Squadra,
            r.Injury_Status,
            r.Availability_Flag,
            int(injury_feed_ok)
        ))

    conn.executemany("""
        INSERT OR IGNORE INTO
        nba_availability_history
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, availability_records)

    total_injury_history = conn.execute("""
        SELECT COUNT(*)
        FROM nba_injury_snapshots
    """).fetchone()[0]

# ============================================================
# 14. ESPORTAZIONE FILE
# ============================================================

stamp = NOW.strftime(
    "%Y%m%d_%H%M%S_%f"
)

export = context.copy()

export["date"] = export["date"].map(
    lambda x: x.isoformat()
)

lineup_export = lineup.copy()

lineup_export["date"] = (
    lineup_export["date"].map(
        lambda x: x.isoformat()
    )
)

injuries.to_csv(
    INJURY_PATH,
    index=False,
    encoding="utf-8-sig"
)

injuries.to_csv(
    os.path.join(
        ARCHIVE_DIR,
        f"NBA_Injuries_{stamp}.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

export.to_csv(
    CSV_PATH + ".tmp",
    index=False,
    encoding="utf-8-sig"
)

os.replace(
    CSV_PATH + ".tmp",
    CSV_PATH
)

export.to_csv(
    os.path.join(
        ARCHIVE_DIR,
        f"NBA_Context_{stamp}.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

impact.to_csv(
    os.path.join(
        BASE_DIR,
        "NBA_Teammate_Impact.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

lineup_export.to_csv(
    os.path.join(
        BASE_DIR,
        "NBA_Estimated_Lineups.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

probability_context.to_csv(
    os.path.join(
        BASE_DIR,
        "NBA_Context_Probabilities.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

status_report = context.groupby(
    [
        "Injury_Status",
        "Availability_Flag"
    ],
    as_index=False
).agg(
    Forecast_Count=("player_id", "size"),
    Unique_Players=("player_id", "nunique")
)

with pd.ExcelWriter(
    OUTPUT_PATH + ".tmp.xlsx",
    engine="openpyxl"
) as writer:

    sheets = {
        "Context Forecast": export,
        "Injury Report": injuries,
        "Estimated Lineups": lineup_export,
        "Rotation Analysis": rotations,
        "Teammate Impact": impact,
        "Context Probabilities": probability_context,
        "Availability Summary": status_report
    }

    for name, data in sheets.items():

        data.to_excel(
            writer,
            sheet_name=name,
            index=False
        )

        sheet = writer.sheets[name]

        sheet.freeze_panes = "A2"
        sheet.auto_filter.ref = sheet.dimensions

os.replace(
    OUTPUT_PATH + ".tmp.xlsx",
    OUTPUT_PATH
)

# ============================================================
# 15. VISUALIZZAZIONE
# ============================================================

print("\nAVAILABILITY SUMMARY")

display(status_report)

fig = px.bar(
    status_report,
    x="Injury_Status",
    y="Forecast_Count",
    color="Injury_Status",
    title="NBA — Player Availability",
    template="plotly_dark",
    color_discrete_map={
        "Available": "#22c55e",
        "Probable": "#84cc16",
        "Questionable": "#eab308",
        "Doubtful": "#f97316",
        "Out": "#ef4444",
        "Unknown": "#64748b"
    }
)

fig.show()

print("\nGIOCATORI DA VERIFICARE")

review = context[
    context["Injury_Status"].isin([
        "Out",
        "Doubtful",
        "Questionable"
    ])
].sort_values(
    "Forecast_PTS",
    ascending=False
)

display(
    review[[
        "Giocatore",
        "Squadra",
        "Avversario",
        "Injury_Status",
        "Injury_Type",
        "Forecast_PTS",
        "Availability_Flag"
    ]].round(2)
)

# ============================================================
# 16. PLAYER CONTEXT EXPLORER
# ============================================================

player_options = context[[
    "player_id",
    "Giocatore"
]].drop_duplicates(
    "player_id"
).sort_values(
    "Giocatore"
)

players = [
    (
        f"{r.Giocatore} ({r.player_id})",
        r.player_id
    )
    for r in player_options.itertuples(
        index=False
    )
]

player_widget = widgets.Dropdown(
    options=players,
    description="Player:"
)

output = widgets.Output()

def show_player(change=None):

    with output:

        clear_output(wait=True)

        selected = context[
            context["player_id"]
            == player_widget.value
        ].sort_values("date")

        if selected.empty:
            return

        r = selected.iloc[0]

        print("=" * 60)
        print(r["Giocatore"].upper())
        print("=" * 60)

        print("Squadra:", r["Squadra"])
        print("Avversario:", r["Avversario"])

        print(
            "Injury Status:",
            r["Injury_Status"]
        )

        print(
            "Availability:",
            r["Availability_Flag"]
        )

        print(
            "Injury Type:",
            r["Injury_Type"]
        )

        print(
            "Ruolo stimato:",
            r["Estimated_Role"]
        )

        print(
            "Minuti ultime 10:",
            round(r["Minutes_Last10"], 2)
        )

        print(
            "Compagni Out:",
            r["Listed_Out_Teammates"]
        )

        print("\nFORECAST")

        rows = []

        for target in TARGETS:

            rows.append({
                "Target": target,
                "Forecast_ML": r[
                    f"Forecast_{target}"
                ],
                "Context_Forecast": r[
                    f"Context_{target}"
                ]
            })

        display(
            pd.DataFrame(rows).round(2)
        )

        player_impact = impact[
            (impact["player_id"] == r["player_id"])
            & (impact["game_id"] == r["game_id"])
        ]

        if not player_impact.empty:

            print("\nIMPATTO STORICO COMPAGNI ASSENTI")

            display(
                player_impact[[
                    "Absent_Player",
                    "Target",
                    "Games_With",
                    "Games_Without",
                    "Mean_With",
                    "Mean_Without",
                    "Historical_Delta"
                ]].round(2)
            )

            fig = px.bar(
                player_impact,
                x="Target",
                y="Historical_Delta",
                color="Absent_Player",
                barmode="group",
                title="Historical Teammate Impact",
                template="plotly_dark"
            )

            fig.add_hline(y=0)
            fig.show()

        else:

            print(
                "\nCampione assenze insufficiente "
                "o nessun compagno Out identificato."
            )

        if not probability_context.empty:

            selected_probs = probability_context[
                (
                    probability_context["player_id"]
                    == r["player_id"]
                )
                & (
                    probability_context["game_id"]
                    == r["game_id"]
                )
            ]

            if not selected_probs.empty:

                print("\nPROBABILITA CONDIZIONATE")

                display(
                    selected_probs[[
                        "Target",
                        "Threshold",
                        "Conditional_Probability_pct"
                    ]].round(2)
                )

player_widget.observe(
    show_player,
    names="value"
)

display(
    player_widget,
    output
)

show_player()

# ============================================================
# 17. RIEPILOGO FINALE
# ============================================================

print("\n" + "=" * 60)
print("STEP 19 COMPLETATO!")
print("INJURY & LINEUP INTELLIGENCE V3.2")
print("=" * 60)

print("Feed ESPN disponibile:", injury_feed_ok)

print(
    "Partite future:",
    context["game_id"].nunique()
)

print(
    "Giocatori:",
    context["player_id"].nunique()
)

print(
    "Forecast esclusi per Out:",
    int(
        (context["Injury_Status"] == "Out").sum()
    )
)

print(
    "Forecast da verificare:",
    int(
        context["Injury_Status"].isin([
            "Questionable", "Doubtful"
        ]).sum()
    )
)

print(
    "Disponibilita sconosciuta:",
    int(
        (context["Injury_Status"] == "Unknown").sum()
    )
)

print(
    "Quintetti stimati - righe:",
    len(lineup)
)

print(
    "Analisi impatto:",
    len(impact)
)

print(
    "Injury snapshot storici:",
    total_injury_history
)

print("\nFILE SALVATI")

print(OUTPUT_PATH)
print(CSV_PATH)
print(INJURY_PATH)

if not injury_feed_ok:

    print(
        "\nATTENZIONE: modulo injury degradato. "
        "Verificare la disponibilita del feed ESPN."
    )

print("\nPRONTO PER STEP 20!")


In [ ]:
# Injury report ingestion


# ============================================================
# STEP 26A — NBA INJURY INTELLIGENCE V2
# ============================================================

# Install with: %pip install pymupdf beautifulsoup4

import os
import requests
import pandas as pd
import fitz
import re
import unicodedata

from pathlib import Path
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

ROOT26 = Path(
    globals().get("BASE_DIR") or os.environ.get("NBA_DATA_DIR") or "/content/drive/MyDrive/NBA_Data_Science"
)

FORECAST26 = ROOT26 / "NBA_Forecast_Engine_V3.csv"

NBA_URL = "https://www.nba.com/players/injury-report"

ESPN_URL = (
    "https://site.api.espn.com/apis/site/v2/"
    "sports/basketball/nba/injuries"
)

session26 = requests.Session()

session26.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "*/*"
})

now26 = datetime.now(timezone.utc)

today_et = now26.astimezone(
    ZoneInfo("America/New_York")
).date()

VALID_STATUS = [
    "Out",
    "Doubtful",
    "Questionable",
    "Probable",
    "Available"
]

# ============================================================
# 1. NORMALIZZAZIONE
# ============================================================

def normalize26(value):

    value = unicodedata.normalize(
        "NFKD", str(value)
    )

    value = "".join(
        c for c in value
        if not unicodedata.combining(c)
    )

    return re.sub(
        r"[^a-z0-9]", "", value.lower()
    )


def status26(value):

    value = str(value).strip().lower()

    mapping = {
        "out": "Out",
        "doubtful": "Doubtful",
        "questionable": "Questionable",
        "probable": "Probable",
        "available": "Available",
        "day-to-day": "Questionable",
        "game time decision": "Questionable"
    }

    return mapping.get(value, "Unknown")


# ============================================================
# 2. CARICAMENTO FORECAST
# ============================================================

forecast26 = pd.read_csv(
    FORECAST26,
    dtype={
        "game_id": str,
        "player_id": str
    }
)

forecast26["date"] = pd.to_datetime(
    forecast26["date"],
    utc=True,
    errors="coerce"
)

forecast26 = forecast26[
    forecast26["date"] > pd.Timestamp(now26)
].copy()

if forecast26.empty:
    raise RuntimeError(
        "Nessun forecast futuro. Aggiornare STEP 17."
    )

forecast26 = forecast26.sort_values(
    "generated_at"
).drop_duplicates(
    ["game_id", "player_id"],
    keep="last"
)

players26 = forecast26[
    ["player_id", "Giocatore", "Squadra"]
].drop_duplicates()

print("Giocatori da verificare:", len(players26))

# ============================================================
# 3. ESPN INJURY REPORT
# ============================================================

espn_records26 = []
espn_ok26 = False

try:

    response = session26.get(
        ESPN_URL,
        timeout=30
    )

    response.raise_for_status()

    payload = response.json()

    groups = payload.get("injuries")

    if not isinstance(groups, list):
        raise ValueError(
            "Struttura ESPN non riconosciuta."
        )

    for group in groups:

        team = group.get(
            "displayName", ""
        )

        for item in group.get("injuries", []):

            athlete = item.get("athlete", {})

            details = item.get("details") or {}

            raw_status = item.get("status", "")

            if isinstance(raw_status, dict):
                raw_status = raw_status.get(
                    "name", ""
                )

            espn_records26.append({
                "player_id": str(
                    athlete.get("id", "")
                ),
                "Giocatore": athlete.get(
                    "displayName", ""
                ),
                "Squadra_ESPN": team,
                "Injury_Status": status26(
                    raw_status
                ),
                "Raw_Status": str(raw_status),
                "Injury_Type": details.get(
                    "type", ""
                ),
                "Injury_Location": details.get(
                    "location", ""
                ),
                "Return_Date": details.get(
                    "returnDate", ""
                ),
                "Comment": item.get(
                    "longComment",
                    item.get("shortComment", "")
                ),
                "Report_Date": item.get(
                    "date", ""
                ),
                "Fetched_At": now26.isoformat(),
                "Source": "ESPN"
            })

    espn_ok26 = bool(espn_records26)

    print(
        "ESPN:",
        len(espn_records26),
        "segnalazioni"
    )

except Exception as e:
    print("ESPN non disponibile:", e)


# ============================================================
# 4. REPORT UFFICIALE NBA
# ============================================================

nba_records26 = []
nba_ok26 = False
official_pdf26 = None
report_time26 = None

try:

    page = session26.get(
        NBA_URL,
        timeout=30
    )

    page.raise_for_status()

    soup = BeautifulSoup(
        page.text,
        "html.parser"
    )

    links = []

    for a in soup.find_all("a", href=True):

        url = urljoin(
            NBA_URL, a["href"]
        )

        match = re.search(
            r"Injury-Report_"
            r"(\d{4}-\d{2}-\d{2})_"
            r"(\d{2})_(\d{2})(AM|PM)",
            url,
            re.I
        )

        if not match:
            continue

        stamp = datetime.strptime(
            f"{match[1]} "
            f"{match[2]}:{match[3]} "
            f"{match[4].upper()}",
            "%Y-%m-%d %I:%M %p"
        ).replace(
            tzinfo=ZoneInfo("America/New_York")
        )

        if (
            stamp.date() == today_et
            and stamp <= now26
            and "ak-static.cms.nba.com" in url
        ):

            links.append((stamp, url))

    if not links:
        raise RuntimeError(
            "Nessun PDF ufficiale odierno trovato."
        )

    report_time26, official_pdf26 = max(
        links,
        key=lambda x: x[0]
    )

    response = session26.get(
        official_pdf26,
        timeout=40
    )

    response.raise_for_status()

    document = fitz.open(
        stream=response.content,
        filetype="pdf"
    )

    text26 = "\n".join(
        page.get_text(sort=True)
        for page in document
    )

    document.close()

    (
        ROOT26 / "NBA_Official_Injury_Report.txt"
    ).write_text(
        text26,
        encoding="utf-8"
    )

    print("Report ufficiale NBA:")
    print(official_pdf26)

    # Riconoscimento dei nomi nel formato
    # Cognome, Nome utilizzato dal report NBA.

    for _, player in players26.iterrows():

        name = player["Giocatore"]

        parts = name.split()

        if len(parts) < 2:
            continue

        variants = []

        for split in range(1, len(parts)):

            first = " ".join(parts[:split])
            last = " ".join(parts[split:])

            variants.append(
                last + ", " + first
            )

        variants.append(name)

        for variant in variants:

            pattern = (
                re.escape(variant)
                + r"\s+"
                + r"(Out|Doubtful|Questionable|"
                  r"Probable|Available)\b"
            )

            match = re.search(
                pattern,
                text26,
                re.I
            )

            if not match:
                continue

            nba_records26.append({
                "player_id": player["player_id"],
                "Giocatore": name,
                "Squadra_ESPN": player["Squadra"],
                "Injury_Status": status26(
                    match.group(1)
                ),
                "Raw_Status": match.group(1),
                "Injury_Type": "",
                "Injury_Location": "",
                "Return_Date": "",
                "Comment": (
                    "Da verificare nel PDF ufficiale"
                ),
                "Report_Date": (
                    report_time26.isoformat()
                ),
                "Fetched_At": now26.isoformat(),
                "Source": "NBA Official"
            })

            break

    nba_ok26 = bool(nba_records26)

    print(
        "NBA Official:",
        len(nba_records26),
        "giocatori riconosciuti"
    )

except Exception as e:
    print("Report NBA non disponibile:", e)


# ============================================================
# 5. UNIFICAZIONE FONTI
# ============================================================

columns26 = [
    "player_id",
    "Giocatore",
    "Squadra_ESPN",
    "Injury_Status",
    "Raw_Status",
    "Injury_Type",
    "Injury_Location",
    "Return_Date",
    "Comment",
    "Report_Date",
    "Fetched_At",
    "Source"
]

injuries26 = pd.DataFrame(
    espn_records26 + nba_records26,
    columns=columns26
)

injuries26 = injuries26[
    injuries26["player_id"].astype(str) != ""
].copy()

injuries26 = injuries26[
    injuries26["Injury_Status"] != "Unknown"
]

# Non sovrascrivere i dati precedenti
# se entrambi i recuperi falliscono.

if injuries26.empty:

    raise RuntimeError(
        "Nessuna segnalazione valida recuperata. "
        "Controllare le fonti e il parser."
    )

# Conservazione di entrambe le fonti

injuries26.to_csv(
    ROOT26 / "NBA_Injury_Sources_V2.csv",
    index=False
)

# ============================================================
# 6. DISPONIBILITA PER SINGOLA PARTITA
# ============================================================

availability26 = []

for _, r in forecast26.iterrows():

    pid = str(r["player_id"])

    player_rows = injuries26[
        injuries26["player_id"].astype(str) == pid
    ]

    team26 = normalize26(r["Squadra"])

    official = player_rows[
        (player_rows["Source"] == "NBA Official")
        & (
            player_rows["Squadra_ESPN"]
            .map(normalize26) == team26
        )
    ]

    espn = player_rows[
        player_rows["Source"] == "ESPN"
    ]

    game_day = r["date"].tz_convert(
        "America/New_York"
    ).date()

    status = "Unknown"
    source = "Not verified"
    note = "Disponibilità non confermata"

    if game_day == today_et:

        candidates = (
            official
            if not official.empty
            else espn
        )

        if not candidates.empty:

            statuses = set(
                candidates["Injury_Status"]
            )

            if len(statuses) == 1:

                status = next(iter(statuses))
                source = candidates.iloc[-1]["Source"]

                note = "Segnalazione recuperata"

                if (
                    not official.empty
                    and not espn.empty
                    and set(espn["Injury_Status"])
                    != statuses
                ):

                    status = "Unknown"
                    source = "NBA / ESPN conflict"
                    note = (
                        "Fonti discordanti: "
                        "verifica manuale necessaria"
                    )

            else:
                note = "Segnalazioni discordanti"

    else:
        note = "Report non verificato per questa data"

    flag = {
        "Out": "EXCLUDE",
        "Doubtful": "HIGH_RISK",
        "Questionable": "UNCERTAIN",
        "Probable": "PROBABLE",
        "Available": "AVAILABLE",
        "Unknown": "UNVERIFIED"
    }[status]

    availability26.append({
        "game_id": r["game_id"],
        "player_id": pid,
        "date": r["date"],
        "Giocatore": r["Giocatore"],
        "Squadra": r["Squadra"],
        "Avversario": r["Avversario"],
        "Injury_Status": status,
        "Availability_Flag": flag,
        "Injury_Source": source,
        "Availability_Note": note,
        "Fetched_At": now26.isoformat()
    })

availability26 = pd.DataFrame(
    availability26
)

availability26.to_csv(
    ROOT26 / "NBA_Availability_V2.csv",
    index=False
)

# ============================================================
# 7. FORECAST OPERATIVO
# ============================================================

context26 = forecast26.drop(
    columns=[
        "Injury_Status",
        "Availability_Flag"
    ],
    errors="ignore"
).merge(
    availability26[
        [
            "game_id",
            "player_id",
            "Injury_Status",
            "Availability_Flag",
            "Injury_Source",
            "Availability_Note"
        ]
    ],
    on=["game_id", "player_id"],
    how="left",
    validate="one_to_one"
)

context26["Roster_Status"] = (
    context26["Availability_Flag"]
)

context26.to_csv(
    ROOT26 / "NBA_Forecast_With_Injuries_V2.csv",
    index=False
)

operational26 = context26[
    context26["Availability_Flag"] != "EXCLUDE"
].copy()

operational26.to_csv(
    ROOT26 / "NBA_Operational_Forecast_V2.csv",
    index=False
)

# ============================================================
# 8. REPORT
# ============================================================

print("\n" + "=" * 60)
print("NBA INJURY INTELLIGENCE V2")
print("=" * 60)

print("ESPN:", "OK" if espn_ok26 else "FAILED")
print("NBA:", "OK" if nba_ok26 else "FAILED")

print("\nStati disponibilità:")

display(
    availability26[
        "Injury_Status"
    ].value_counts()
)

print("\nGiocatori OUT:")

display(
    availability26[
        availability26["Injury_Status"] == "Out"
    ]
)

print("\nCONTROLLO LAMELO BALL")

display(
    availability26[
        availability26["Giocatore"].str.contains(
            "LaMelo Ball",
            case=False,
            na=False
        )
    ]
)

print("\nFile salvati su Google Drive:")

for file in [
    "NBA_Injury_Sources_V2.csv",
    "NBA_Availability_V2.csv",
    "NBA_Forecast_With_Injuries_V2.csv",
    "NBA_Operational_Forecast_V2.csv"
]:
    print(file)

print("\nSTEP 26A COMPLETATO")


In [ ]:
# Contextual injury integration


# ============================================================
# STEP 26B — INJURY INTEGRATION & DASHBOARD RESTART
# ============================================================

from google.colab import drive
from pathlib import Path
from datetime import datetime
import pandas as pd
import shutil

# Use the directory selected by the setup cell.
# This stage modifies generated forecast files and creates its own backups.
import os
if not globals().get("BASE_DIR") and not os.environ.get("NBA_DATA_DIR"):
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

ROOT = Path(
    globals().get("BASE_DIR") or os.environ.get("NBA_DATA_DIR") or "/content/drive/MyDrive/NBA_Data_Science"
)

BACKUP = ROOT / "Backup_Injury_V2" / datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

BACKUP.mkdir(parents=True, exist_ok=True)

# ============================================================
# 1. CARICAMENTO NUOVI DATI
# ============================================================

keys = ["game_id", "player_id"]

availability = pd.read_csv(
    ROOT / "NBA_Availability_V2.csv",
    dtype=str
)

injuries = pd.read_csv(
    ROOT / "NBA_Injury_Sources_V2.csv",
    dtype=str
)

availability = availability.drop_duplicates(
    keys, keep="last"
)

columns = [
    "Injury_Status",
    "Availability_Flag",
    "Injury_Source",
    "Availability_Note"
]

def backup_file(path):
    if path.exists():
        shutil.copy2(path, BACKUP / path.name)

def save_csv(df, path):
    backup_file(path)
    temp = path.with_suffix(".tmp")
    df.to_csv(temp, index=False)
    temp.replace(path)

def integrate_availability(df):

    df = df.drop(
        columns=columns,
        errors="ignore"
    )

    df = df.merge(
        availability[keys + columns],
        on=keys,
        how="left",
        validate="many_to_one"
    )

    df["Injury_Status"] = (
        df["Injury_Status"].fillna("Unknown")
    )

    df["Availability_Flag"] = (
        df["Availability_Flag"].fillna("UNVERIFIED")
    )

    df["Roster_Status"] = df["Availability_Flag"]

    return df

# ============================================================
# 2. AGGIORNAMENTO INJURY REPORT
# ============================================================

save_csv(
    injuries,
    ROOT / "NBA_Injury_Report.csv"
)

# ============================================================
# 3. AGGIORNAMENTO FORECAST CONTESTUALE
# ============================================================

raw = pd.read_csv(
    ROOT / "NBA_Forecast_Engine_V3.csv",
    dtype={k: str for k in keys}
)

context_path = ROOT / "NBA_Context_Aware_Forecast.csv"

context = raw.copy()

if context_path.exists():

    previous = pd.read_csv(
        context_path,
        dtype={k: str for k in keys}
    )

    join_keys = keys + ["generated_at"]

    if all(k in previous.columns for k in join_keys):

        extra = [
            c for c in previous.columns
            if c not in raw.columns
            and c not in columns
        ]

        context = raw.merge(
            previous[join_keys + extra].drop_duplicates(
                join_keys, keep="last"
            ),
            on=join_keys,
            how="left"
        )

context = integrate_availability(context)

save_csv(context, context_path)

# ============================================================
# 4. AGGIORNAMENTO PROBABILITA
# ============================================================

prob = pd.read_csv(
    ROOT / "NBA_Probability_Engine_V3.csv",
    dtype={k: str for k in keys}
)

prob = integrate_availability(prob)

prob = prob[
    prob["Injury_Status"] != "Out"
].copy()

save_csv(
    prob,
    ROOT / "NBA_Context_Probabilities.csv"
)

# ============================================================
# 5. AGGIORNAMENTO FORMAZIONI
# ============================================================

lineup_path = ROOT / "NBA_Estimated_Lineups.csv"

if lineup_path.exists():

    lineup = pd.read_csv(
        lineup_path,
        dtype={k: str for k in keys}
    )

    if all(k in lineup.columns for k in keys):

        lineup = integrate_availability(lineup)

        lineup = lineup[
            lineup["Injury_Status"] != "Out"
        ].copy()

        save_csv(lineup, lineup_path)

# ============================================================
# 6. CONTROLLO FINALE
# ============================================================

print("=" * 60)
print("INJURY INTELLIGENCE V2 INTEGRATO")
print("=" * 60)

print("\nStati giocatori:")

display(
    context["Injury_Status"].value_counts()
)

print("\nGiocatori OUT:")

display(
    context.loc[
        context["Injury_Status"] == "Out",
        ["Giocatore", "Squadra", "Injury_Source"]
    ]
)

print("\nCONTROLLO LAMELO BALL:")

display(
    context.loc[
        context["Giocatore"].str.contains(
            "LaMelo Ball",
            case=False,
            na=False
        ),
        [
            "Giocatore",
            "Injury_Status",
            "Availability_Flag",
            "Injury_Source",
            "Forecast_PTS"
        ]
    ]
)

print("\nBackup:", BACKUP)
print("\nSTEP 26B COMPLETATO")
